# MLOps и продакшен - как довести модель до пользователя

**Автор:** ML Instructor
**Уровень:** intermediate (PyTorch basics + обучение моделей)
**Время выполнения:** ~120-150 минут

---

### Чему вы научитесь

После выполнения этого ноутбука вы:
- поймёте **что** такое MLOps и почему модели падают в продакшене;
- освоите **сериализацию моделей** (torch.save/load, state_dict);
- создадите **FastAPI продакшен-сервер** с /predict, /health, /info;
- реализуете **систему версионирования** моделей;
- построите **A/B тестирование** с двумя моделями и маршрутизацией;
- соберёте **мониторинг-дашборд** (латентность, ошибки, распределение);
- создадите **интерактивный виджет** для настройки деплоя;
- реализуете **детекцию дрифта данных** статистическими тестами;
- автоматизируете **переобучение** при дрифте (drift -> retrain -> deploy);
- проведёте **нагрузочное тестирование** с метриками;
- изучите **ландшафт MLOps-инструментов**;
- подведёте **итоги всего курса** из 15 ноутбуков.

### Принцип этого ноутбука

> Вы - автор, а не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Setup | Установка и импорт библиотек |
| 2 | Что такое MLOps? | Жизненный цикл ML, почему модели падают в продакшене |
| 3 | Сериализация моделей | torch.save/load, state_dict, полностью запускаемый |
| 4 | FastAPI продакшен-сервер | /predict с Pydantic, /health, /info, ngrok |
| 5 | Версионирование моделей | Простая система версий, запускаемая |
| 6 | A/B тестирование сервер | Маршрутизация к модели A или B, /metrics, /set_ratio, ngrok |
| 7 | Мониторинг-дашборд | Латентность, ошибки, распределение предсказаний |
| 8 | Интерактивный деплой-эксплорер | Виджеты: batch_size, workers, timeout, model_version |
| 9 | Детекция дрифта данных | Статистические тесты, запускаемая |
| 10 | Автоматическое переобучение | drift -> retrain -> deploy, запускаемая |
| 11 | Нагрузочное тестирование | Запускаемое с метриками |
| 12 | Ландшафт MLOps-инструментов | Сравнительная таблица и диаграмма |
| 13 | Практика + ИТОГИ КУРСА | Задания и подведение итогов 15 ноутбуков |

---

---
## Шаг 1. Setup - установка и импорт библиотек

| Библиотека | Зачем |
|------------|-------|
| **torch** | Создание и обучение моделей |
| **fastapi** | Создание продакшен API сервера |
| **uvicorn** | ASGI-сервер для FastAPI |
| **pydantic** | Валидация входных данных API |
| **pyngrok** | Публичный доступ к серверу из Colab |
| **ipywidgets** | Интерактивные виджеты |
| **scipy** | Статистические тесты для детекции дрифта |
| **matplotlib** | Визуализация метрик и графиков |
| **numpy** | Математические операции и массивы |

In [ ]:
# ============================================================
# ШАГ 1: Устанавливаем необходимые библиотеки
# ============================================================

!pip install -q fastapi uvicorn pyngrok pydantic  # серверные библиотеки
!pip install -q scipy                               # статистические тесты
!pip install -q torch torchvision                    # ML фреймворк (обычно уже есть в Colab)

print("Все библиотеки установлены!")

In [ ]:
# ============================================================
# ШАГ 1 (продолжение): Импортируем все нужные модули
# ============================================================

import numpy as np                                  # основная библиотека для массивов и математики
import matplotlib.pyplot as plt                     # для построения графиков и визуализаций
import torch                                        # PyTorch - основной фреймворк
import torch.nn as nn                               # слои нейросети (Linear, Conv2d...)
import torch.nn.functional as F                     # функции активации, функции потерь
import torch.optim as optim                         # оптимизаторы (Adam, SGD...)
from torch.utils.data import DataLoader, TensorDataset  # утилиты загрузки данных
import json                                         # для работы с JSON форматом
import os                                           # для файловых операций
import time                                         # для замеров времени
import threading                                    # для запуска сервера в фоне
import copy                                         # для глубокого копирования моделей
import uuid                                         # для генерации уникальных ID
from collections import defaultdict                 # словарь со значением по умолчанию
from datetime import datetime                       # для работы с датой и временем
from IPython.display import display, HTML, clear_output  # красивый вывод в ноутбуке
import ipywidgets as widgets                        # интерактивные виджеты
from ipywidgets import interact, interactive, fixed, Layout  # декораторы виджетов
from scipy import stats                             # статистические тесты

# Настройка matplotlib для лучшего отображения
plt.rcParams['figure.figsize'] = (12, 7)           # размер фигуры по умолчанию
plt.rcParams['font.size'] = 12                     # размер шрифта
plt.rcParams['axes.grid'] = True                   # показывать сетку

# Фиксируем случайные种子 для воспроизводимости
torch.manual_seed(42)                              #种子 для PyTorch
np.random.seed(42)                                 #种子 для NumPy

# Определяем устройство: GPU если доступно, иначе CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU или CPU

print(f"PyTorch: {torch.__version__}")             # версия PyTorch
print(f"Устройство: {device}")                     # выбранное устройство
print("Все модули импортированы!")

---
## Шаг 2. Что такое MLOps?

**MLOps** (Machine Learning Operations) - это набор практик для надёжного и эффективного развёртывания, мониторинга и управления ML-моделями в продакшене.

### Почему модели падают в продакшене?

| Причина | Описание | Частота |
|---------|----------|---------|
| **Data drift** | Распределение входных данных меняется со временем | 45% |
| **Concept drift** | Связь признаки->целевая переменная меняется | 30% |
| **Инфраструктурные сбои** | Сервер падает, не хватает памяти, таймауты | 15% |
| **Ошибки в пайплайне** | Баги в предобработке, неверный формат данных | 10% |

### Жизненный цикл ML-модели

```
Сбор данных -> Обучение -> Оценка -> Сериализация -> Деплой -> Мониторинг -> Переобучение
     ^                                                                          |
     |__________________________________________________________________________|
```

### MLOps уровни зрелости

| Уровень | Описание | Автоматизация |
|---------|----------|---------------|
| 0 | Ручной процесс | Нет |
| 1 | ML pipeline автоматизирован | Частичная |
| 2 | CI/CD для ML | Полная |

In [ ]:
# ============================================================
# ШАГ 2: Визуализируем жизненный цикл ML-модели
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(16, 8))     # создаём фигуру

# Этапы жизненного цикла
stages = [                                          # названия этапов
    'Сбор\nданных',
    'Обучение\nмодели',
    'Оценка\nкачества',
    'Сериализация',
    'Деплой',
    'Мониторинг',
    'Детекция\nдрифта',
    'Пере-\nобучение'
]

# Цвета для каждого этапа
colors = [                                          # цвета этапов
    '#2196F3',                                      # синий - данные
    '#4CAF50',                                      # зелёный - обучение
    '#FF9800',                                      # оранжевый - оценка
    '#9C27B0',                                      # фиолетовый - сериализация
    '#F44336',                                      # красный - деплой
    '#00BCD4',                                      # голубой - мониторинг
    '#FF5722',                                      # тёмно-оранжевый - дрифт
    '#795548',                                      # коричневый - переобучение
]

n = len(stages)                                     # количество этапов
radius = 3.5                                        # радиус окружности
center_x, center_y = 5, 5                          # центр окружности

# Рисуем этапы по кругу
for i in range(n):                                  # для каждого этапа
    angle = 2 * np.pi * i / n - np.pi / 2          # угол для расположения
    x = center_x + radius * np.cos(angle)           # координата x
    y = center_y + radius * np.sin(angle)           # координата y

    circle = plt.Circle((x, y), 0.8,               # круг для этапа
                        facecolor=colors[i],        # цвет заливки
                        edgecolor='black',           # цвет границы
                        linewidth=2,                 # толщина границы
                        alpha=0.8)                   # прозрачность
    ax.add_patch(circle)                             # добавляем круг
    ax.text(x, y, stages[i],                        # текст этапа
            ha='center', va='center',               # выравнивание
            fontsize=9, fontweight='bold',          # размер и стиль шрифта
            color='white')                          # цвет текста

    # Рисуем стрелку к следующему этапу
    next_i = (i + 1) % n                            # индекс следующего этапа
    next_angle = 2 * np.pi * next_i / n - np.pi / 2  # угол следующего
    nx = center_x + radius * np.cos(next_angle)     # x следующего
    ny = center_y + radius * np.sin(next_angle)     # y следующего

    # Направление стрелки (от края текущего круга к краю следующего)
    dx = nx - x                                     # разница x
    dy = ny - y                                     # разница y
    length = np.sqrt(dx**2 + dy**2)                 # длина вектора
    # Сдвигаем начало и конец стрелки от центров кругов
    start_x = x + 0.85 * dx / length               # начало x
    start_y = y + 0.85 * dy / length               # начало y
    end_x = nx - 0.85 * dx / length                # конец x
    end_y = ny - 0.85 * dy / length                # конец y

    ax.annotate('', xy=(end_x, end_y),              # стрелка
                xytext=(start_x, start_y),
                arrowprops=dict(arrowstyle='->',     # стиль стрелки
                               lw=2,                 # толщина
                               color='#333333'))     # цвет

# Заголовок в центре
ax.text(center_x, center_y, 'MLOps\nLifecycle',   # заголовок
        ha='center', va='center',                   # выравнивание
        fontsize=14, fontweight='bold',             # стиль
        bbox=dict(boxstyle='round,pad=0.3',         # рамка
                 facecolor='lightyellow',           # цвет фона
                 edgecolor='black'))                # цвет границы

ax.set_xlim(0, 10)                                 # пределы x
ax.set_ylim(0, 10)                                 # пределы y
ax.set_aspect('equal')                              # равные масштабы
ax.set_title('Жизненный цикл ML-модели в MLOps', fontsize=16, fontweight='bold')  # заголовок
ax.axis('off')                                      # скрываем оси

plt.tight_layout()                                   # подгоняем layout
plt.show()                                           # отображаем

In [ ]:
# ============================================================
# ШАГ 2 (продолжение): Статистика почему модели падают
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))    # два подграфика

# --- Левый график: причины провалов ---
reasons = ['Data drift', 'Concept drift', 'Инфраструктура', 'Ошибки пайплайна']  # причины
percentages = [45, 30, 15, 10]                      # проценты
colors_bar = ['#F44336', '#FF9800', '#2196F3', '#9C27B0']  # цвета

bars = axes[0].barh(reasons, percentages,           # горизонтальная диаграмма
                    color=colors_bar, edgecolor='black')  # цвета и границы
for bar, pct in zip(bars, percentages):             # добавляем проценты
    axes[0].text(bar.get_width() + 1,              # позиция x
                bar.get_y() + bar.get_height()/2,   # позиция y
                f'{pct}%',                          # текст
                va='center', fontsize=12, fontweight='bold')  # стиль

axes[0].set_xlim(0, 55)                             # пределы x
axes[0].set_xlabel('Процент случаев (%)', fontsize=12)  # подпись оси
axes[0].set_title('Почему модели падают в продакшене', fontsize=14, fontweight='bold')  # заголовок

# --- Правый график: уровни зрелости MLOps ---
levels = ['Уровень 0\n(Ручной)', 'Уровень 1\n(Pipeline)', 'Уровень 2\n(CI/CD)']  # уровни
maturity_scores = [20, 60, 95]                      # оценки зрелости
colors_mat = ['#F44336', '#FF9800', '#4CAF50']      # цвета

bars2 = axes[1].bar(levels, maturity_scores,        # вертикальная диаграмма
                    color=colors_mat, edgecolor='black', width=0.5)  # стиль
for bar, score in zip(bars2, maturity_scores):      # добавляем значения
    axes[1].text(bar.get_x() + bar.get_width()/2,   # позиция x
                bar.get_height() + 2,               # позиция y
                f'{score}%',                        # текст
                ha='center', fontsize=14, fontweight='bold')  # стиль

axes[1].set_ylim(0, 110)                            # пределы y
axes[1].set_ylabel('Автоматизация (%)', fontsize=12)  # подпись оси
axes[1].set_title('Уровни зрелости MLOps', fontsize=14, fontweight='bold')  # заголовок

plt.tight_layout()                                   # подгоняем layout
plt.show()                                           # отображаем

---
## Шаг 3. Сериализация моделей

**Сериализация** - это сохранение обученной модели на диск для последующей загрузки и использования.

### Способы сериализации в PyTorch

| Способ | Что сохраняет | Плюсы | Минусы |
|--------|--------------|-------|--------|
| `torch.save(model, ...)` | Весь объект модели | Просто | Зависит от структуры классов |
| `torch.save(state_dict, ...)` | Только веса | Гибко, безопасно | Нужно создавать модель при загрузке |
| ONNX | Универсальный формат | Кросс-фреймворк | Не все операции поддерживаются |
| TorchScript | Оптимизированный граф | Быстрый inference | Ограниченная динамика |

> В продакшене предпочтительно использовать **state_dict** - вы контролируете архитектуру модели отдельно от весов.

In [ ]:
# ============================================================
# ШАГ 3: Создаём простую модель для демонстрации сериализации
# ============================================================

# Создаём простую нейросеть для классификации
class SimpleClassifier(nn.Module):                   # простой классификатор
    def __init__(self, input_dim=20, hidden_dim=64, num_classes=3):  # конструктор
        super(SimpleClassifier, self).__init__()     # вызываем родительский конструктор
        self.fc1 = nn.Linear(input_dim, hidden_dim)  # первый полносвязный слой
        self.fc2 = nn.Linear(hidden_dim, 32)         # второй полносвязный слой
        self.fc3 = nn.Linear(32, num_classes)        # выходной слой
        self.dropout = nn.Dropout(0.2)               # dropout для регуляризации

    def forward(self, x):                            # прямой проход
        x = F.relu(self.fc1(x))                     # ReLU после первого слоя
        x = self.dropout(x)                          # dropout
        x = F.relu(self.fc2(x))                     # ReLU после второго слоя
        x = self.fc3(x)                              # выходной слой (логиты)
        return x                                     # возвращаем логиты

# Создаём модель и обучаем на синтетических данных
model_v1 = SimpleClassifier(input_dim=20, hidden_dim=64, num_classes=3).to(device)  # модель v1

# Генерируем синтетические данные для обучения
n_samples = 1000                                     # количество примеров
X_train = torch.randn(n_samples, 20).to(device)     # признаки - случайные
y_train = torch.randint(0, 3, (n_samples,)).to(device)  # метки - 3 класса

# Обучаем модель
optimizer = optim.Adam(model_v1.parameters(), lr=0.01)  # оптимизатор Adam
criterion = nn.CrossEntropyLoss()                    # функция потерь

model_v1.train()                                     # режим обучения
for epoch in range(50):                              # 50 эпох
    optimizer.zero_grad()                            # обнуляем градиенты
    outputs = model_v1(X_train)                     # прямой проход
    loss = criterion(outputs, y_train)               # вычисляем loss
    loss.backward()                                  # обратное распространение
    optimizer.step()                                 # шаг оптимизации

# Проверяем точность
model_v1.eval()                                      # режим оценки
with torch.no_grad():                                # без градиентов
    outputs = model_v1(X_train)                     # прямой проход
    preds = outputs.argmax(dim=1)                    # предсказания
    accuracy = (preds == y_train).float().mean().item()  # точность

print(f"Модель v1 обучена! Точность: {accuracy:.4f}")  # результат

In [ ]:
# ============================================================
# ШАГ 3 (продолжение): Сериализация через state_dict
# ============================================================

# Создаём директорию для моделей
os.makedirs('models', exist_ok=True)                # создаём папку models

# --- Способ 1: Сохраняем только state_dict (рекомендуемый) ---
torch.save(model_v1.state_dict(), 'models/model_v1_state_dict.pt')  # сохраняем веса
print("state_dict сохранён в models/model_v1_state_dict.pt")  # подтверждение

# --- Способ 2: Сохраняем всю модель (проще, но менее гибко) ---
torch.save(model_v1, 'models/model_v1_full.pt')    # сохраняем весь объект
print("Полная модель сохранена в models/model_v1_full.pt")  # подтверждение

# --- Способ 3: Сохраняем с метаданными (продакшен-подход) ---
checkpoint = {                                      # словарь чекпоинта
    'model_state_dict': model_v1.state_dict(),      # веса модели
    'optimizer_state_dict': optimizer.state_dict(),  # состояние оптимизатора
    'epoch': 50,                                    # номер эпохи
    'loss': loss.item(),                            # финальный loss
    'input_dim': 20,                                # размер входа
    'hidden_dim': 64,                               # размер скрытого слоя
    'num_classes': 3,                               # количество классов
    'timestamp': datetime.now().isoformat(),         # время сохранения
    'version': '1.0.0',                             # версия модели
}
torch.save(checkpoint, 'models/model_v1_checkpoint.pt')  # сохраняем чекпоинт
print("Чекпоинт сохранён в models/model_v1_checkpoint.pt")  # подтверждение

# Показываем размер файлов
for f in os.listdir('models'):                      # перебираем файлы
    size = os.path.getsize(f'models/{f}')           # размер файла
    print(f"  {f}: {size / 1024:.1f} KB")          # выводим размер

In [ ]:
# ============================================================
# ШАГ 3 (продолжение): Загрузка модели из разных форматов
# ============================================================

# --- Загрузка из state_dict ---
model_loaded_sd = SimpleClassifier(input_dim=20, hidden_dim=64, num_classes=3).to(device)  # создаём архитектуру
model_loaded_sd.load_state_dict(torch.load('models/model_v1_state_dict.pt', map_location=device))  # загружаем веса
model_loaded_sd.eval()                               # режим оценки
print("Модель загружена из state_dict!")            # подтверждение

# --- Загрузка полной модели ---
model_loaded_full = torch.load('models/model_v1_full.pt', map_location=device)  # загружаем целиком
model_loaded_full.eval()                             # режим оценки
print("Полная модель загружена!")                   # подтверждение

# --- Загрузка из чекпоинта ---
ckpt = torch.load('models/model_v1_checkpoint.pt', map_location=device)  # загружаем чекпоинт
model_loaded_ckpt = SimpleClassifier(               # создаём модель с параметрами из чекпоинта
    input_dim=ckpt['input_dim'],                    # размер входа
    hidden_dim=ckpt['hidden_dim'],                  # размер скрытого слоя
    num_classes=ckpt['num_classes']                 # количество классов
).to(device)
model_loaded_ckpt.load_state_dict(ckpt['model_state_dict'])  # загружаем веса
model_loaded_ckpt.eval()                             # режим оценки
print(f"Чекпоинт загружен! Версия: {ckpt['version']}, Эпоха: {ckpt['epoch']}")  # информация

# Проверяем, что все модели дают одинаковые предсказания
test_input = torch.randn(1, 20).to(device)          # тестовый вход
with torch.no_grad():                                # без градиентов
    pred1 = model_loaded_sd(test_input).argmax(dim=1).item()   # предсказание 1
    pred2 = model_loaded_full(test_input).argmax(dim=1).item()  # предсказание 2
    pred3 = model_loaded_ckpt(test_input).argmax(dim=1).item()  # предсказание 3

print(f"\nПредсказания совпадают: {pred1 == pred2 == pred3}")  # проверка
print(f"  state_dict: класс {pred1}")               # результат 1
print(f"  full model: класс {pred2}")               # результат 2
print(f"  checkpoint: класс {pred3}")               # результат 3

---
## Шаг 4. FastAPI продакшен-сервер

Создадим полноценный API-сервер для обслуживания ML-модели с:
- **/predict** - предсказание с валидацией через Pydantic
- **/health** - проверка здоровья сервера
- **/info** - информация о модели
- **/batch_predict** - пакетное предсказание

> Это первый из двух FastAPI серверов - продакшен API.

In [ ]:
# ============================================================
# ШАГ 4: Создаём вторую версию модели для A/B тестирования
# ============================================================

# Обучаем модель v2 с другой архитектурой (больше нейронов)
class ImprovedClassifier(nn.Module):                 # улучшенный классификатор
    def __init__(self, input_dim=20, hidden_dim=128, num_classes=3):  # конструктор
        super(ImprovedClassifier, self).__init__()   # вызываем родительский конструктор
        self.fc1 = nn.Linear(input_dim, hidden_dim)  # первый слой (128 нейронов)
        self.bn1 = nn.BatchNorm1d(hidden_dim)        # batch normalization
        self.fc2 = nn.Linear(hidden_dim, 64)         # второй слой
        self.bn2 = nn.BatchNorm1d(64)                # batch normalization
        self.fc3 = nn.Linear(64, 32)                 # третий слой
        self.fc4 = nn.Linear(32, num_classes)        # выходной слой
        self.dropout = nn.Dropout(0.3)               # dropout

    def forward(self, x):                            # прямой проход
        x = self.dropout(F.relu(self.bn1(self.fc1(x))))  # слой 1 + BN + ReLU + dropout
        x = self.dropout(F.relu(self.bn2(self.fc2(x))))  # слой 2 + BN + ReLU + dropout
        x = F.relu(self.fc3(x))                     # слой 3 + ReLU
        x = self.fc4(x)                              # выходной слой
        return x                                     # возвращаем логиты

# Создаём и обучаем модель v2
model_v2 = ImprovedClassifier(input_dim=20, hidden_dim=128, num_classes=3).to(device)  # модель v2
optimizer2 = optim.Adam(model_v2.parameters(), lr=0.005)  # оптимизатор для v2

model_v2.train()                                     # режим обучения
for epoch in range(80):                              # 80 эпох (больше для более глубокой сети)
    optimizer2.zero_grad()                           # обнуляем градиенты
    outputs = model_v2(X_train)                     # прямой проход
    loss = criterion(outputs, y_train)               # loss
    loss.backward()                                  # обратное распространение
    optimizer2.step()                                # шаг оптимизации

# Сохраняем модель v2
checkpoint_v2 = {                                    # чекпоинт v2
    'model_state_dict': model_v2.state_dict(),      # веса модели
    'input_dim': 20,                                # размер входа
    'hidden_dim': 128,                              # размер скрытого слоя
    'num_classes': 3,                               # количество классов
    'version': '2.0.0',                             # версия модели
    'timestamp': datetime.now().isoformat(),         # время сохранения
}
torch.save(checkpoint_v2, 'models/model_v2_checkpoint.pt')  # сохраняем

# Проверяем точность v2
model_v2.eval()                                      # режим оценки
with torch.no_grad():                                # без градиентов
    outputs2 = model_v2(X_train)                    # прямой проход
    preds2 = outputs2.argmax(dim=1)                  # предсказания
    accuracy2 = (preds2 == y_train).float().mean().item()  # точность

print(f"Модель v2 обучена! Точность: {accuracy2:.4f}")  # результат
print(f"Модель v1 точность: {accuracy:.4f}")        # сравнение с v1

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Создаём FastAPI продакшен-сервер
# ============================================================

import uvicorn                                       # ASGI-сервер
from fastapi import FastAPI, HTTPException           # фреймворк и исключения
from pydantic import BaseModel, Field, confloat      # валидация данных
from typing import List, Optional                    # типы для аннотаций

# Определяем Pydantic модели для валидации входных данных
class PredictionRequest(BaseModel):                  # запрос на предсказание
    features: List[float] = Field(..., min_length=20, max_length=20)  # ровно 20 признаков

class BatchPredictionRequest(BaseModel):             # запрос на пакетное предсказание
    batch: List[List[float]] = Field(..., min_length=1, max_length=100)  # до 100 примеров

class PredictionResponse(BaseModel):                 # ответ предсказания
    predicted_class: int                             # предсказанный класс
    confidence: float                                # уверенность
    probabilities: List[float]                       # вероятности всех классов
    model_version: str                               # версия модели
    prediction_id: str                               # уникальный ID предсказания
    latency_ms: float                                # время ответа в мс

class HealthResponse(BaseModel):                     # ответ проверки здоровья
    status: str                                      # статус
    model_loaded: bool                               # загружена ли модель
    uptime_seconds: float                            # время работы

class InfoResponse(BaseModel):                       # ответ информации о модели
    model_name: str                                  # имя модели
    version: str                                     # версия
    input_dim: int                                   # размер входа
    num_classes: int                                 # количество классов
    total_predictions: int                           # всего предсказаний

# Создаём FastAPI приложение
app = FastAPI(                                       # приложение
    title="ML Production API",                      # заголовок
    version="1.0.0",                                # версия
    description="Продакшен API для ML-модели классификации"  # описание
)

# Глобальное состояние сервера
server_state = {                                     # состояние сервера
    'model': model_v1,                              # текущая модель
    'model_version': '1.0.0',                       # версия модели
    'total_predictions': 0,                         # счётчик предсказаний
    'start_time': time.time(),                      # время запуска
    'latencies': [],                                # список латентностей
    'errors': [],                                   # список ошибок
}

# Эндпоинт: предсказание
@app.post("/predict", response_model=PredictionResponse)  # POST /predict
async def predict(request: PredictionRequest):       # функция предсказания
    start = time.time()                              # замеряем время
    try:                                             # обрабатываем ошибки
        # Конвертируем входные данные в тензор
        x = torch.tensor([request.features], dtype=torch.float32).to(device)  # тензор

        # Получаем предсказание
        server_state['model'].eval()                 # режим оценки
        with torch.no_grad():                        # без градиентов
            logits = server_state['model'](x)        # прямой проход
            probs = F.softmax(logits, dim=1)         # вероятности
            pred_class = probs.argmax(dim=1).item()  # класс
            confidence = probs.max().item()           # уверенность

        latency = (time.time() - start) * 1000      # латентность в мс

        # Обновляем статистику
        server_state['total_predictions'] += 1       # увеличиваем счётчик
        server_state['latencies'].append(latency)    # сохраняем латентность

        return PredictionResponse(                   # возвращаем ответ
            predicted_class=pred_class,              # класс
            confidence=round(confidence, 4),         # уверенность
            probabilities=[round(p, 4) for p in probs[0].cpu().tolist()],  # вероятности
            model_version=server_state['model_version'],  # версия
            prediction_id=str(uuid.uuid4())[:8],    # ID предсказания
            latency_ms=round(latency, 2)             # латентность
        )
    except Exception as e:                           # если ошибка
        server_state['errors'].append(str(e))        # записываем ошибку
        raise HTTPException(status_code=500, detail=str(e))  # возвращаем 500

# Эндпоинт: проверка здоровья
@app.get("/health", response_model=HealthResponse)  # GET /health
async def health():                                  # функция проверки
    uptime = time.time() - server_state['start_time']  # время работы
    return HealthResponse(                           # возвращаем статус
        status="healthy",                            # статус
        model_loaded=server_state['model'] is not None,  # модель загружена
        uptime_seconds=round(uptime, 1)             # время работы
    )

# Эндпоинт: информация о модели
@app.get("/info", response_model=InfoResponse)      # GET /info
async def info():                                    # функция информации
    return InfoResponse(                             # возвращаем информацию
        model_name="SimpleClassifier",               # имя модели
        version=server_state['model_version'],       # версия
        input_dim=20,                                # размер входа
        num_classes=3,                               # количество классов
        total_predictions=server_state['total_predictions']  # всего предсказаний
    )

# Эндпоинт: пакетное предсказание
@app.post("/batch_predict")                          # POST /batch_predict
async def batch_predict(request: BatchPredictionRequest):  # функция
    start = time.time()                              # замеряем время
    try:                                             # обрабатываем ошибки
        x = torch.tensor(request.batch, dtype=torch.float32).to(device)  # тензор
        server_state['model'].eval()                 # режим оценки
        with torch.no_grad():                        # без градиентов
            logits = server_state['model'](x)        # прямой проход
            probs = F.softmax(logits, dim=1)         # вероятности
            pred_classes = probs.argmax(dim=1).cpu().tolist()  # классы

        latency = (time.time() - start) * 1000      # латентность
        server_state['total_predictions'] += len(request.batch)  # обновляем счётчик

        return {                                     # возвращаем результат
            "predictions": pred_classes,             # список классов
            "batch_size": len(request.batch),        # размер батча
            "latency_ms": round(latency, 2),         # латентность
            "model_version": server_state['model_version']  # версия
        }
    except Exception as e:                           # если ошибка
        raise HTTPException(status_code=500, detail=str(e))  # возвращаем 500

print("FastAPI продакшен-сервер создан!")            # подтверждение
print("Эндпоинты: /predict, /health, /info, /batch_predict")  # список

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Запускаем продакшен-сервер с ngrok
# ============================================================

from pyngrok import ngrok                            # импортируем ngrok

# Раскомментируйте и добавьте ваш токен ngrok:
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")           # токен ngrok

# Запускаем сервер в фоновом потоке
def run_production_server():                         # функция запуска сервера
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")  # запускаем uvicorn

server_thread = threading.Thread(target=run_production_server, daemon=True)  # поток
server_thread.start()                                # запускаем поток
time.sleep(2)                                        # ждём запуска

# Запускаем ngrok для публичного доступа
try:                                                 # пытаемся запустить ngrok
    public_url = ngrok.connect(8000)                 # открываем туннель на порт 8000
    print(f"Продакшен-сервер запущен!")              # подтверждение
    print(f"Публичный URL: {public_url}")            # публичный URL
    print(f"Документация: {public_url}/docs")        # Swagger UI
except Exception as e:                               # если ошибка ngrok
    print(f"ngrok не запущен: {e}")                  # сообщение об ошибке
    print("Сервер работает локально на http://localhost:8000")  # локальный доступ

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Тестируем продакшен-сервер
# ============================================================

import requests                                      # для HTTP-запросов

base_url = "http://localhost:8000"                   # базовый URL сервера

# Тест 1: Проверка здоровья
try:                                                 # обрабатываем ошибки
    health_resp = requests.get(f"{base_url}/health")  # GET /health
    print("=== /health ===")                         # заголовок
    print(json.dumps(health_resp.json(), indent=2))  # выводим JSON
except Exception as e:                               # если ошибка
    print(f"Ошибка /health: {e}")                    # сообщение

# Тест 2: Информация о модели
try:                                                 # обрабатываем ошибки
    info_resp = requests.get(f"{base_url}/info")     # GET /info
    print("\n=== /info ===")                        # заголовок
    print(json.dumps(info_resp.json(), indent=2))    # выводим JSON
except Exception as e:                               # если ошибка
    print(f"Ошибка /info: {e}")                      # сообщение

# Тест 3: Предсказание
try:                                                 # обрабатываем ошибки
    test_features = np.random.randn(20).tolist()    # генерируем 20 признаков
    predict_resp = requests.post(                    # POST /predict
        f"{base_url}/predict",
        json={"features": test_features}
    )
    print("\n=== /predict ===")                     # заголовок
    print(json.dumps(predict_resp.json(), indent=2))  # выводим JSON
except Exception as e:                               # если ошибка
    print(f"Ошибка /predict: {e}")                   # сообщение

# Тест 4: Пакетное предсказание
try:                                                 # обрабатываем ошибки
    batch_features = np.random.randn(5, 20).tolist()  # 5 примеров по 20 признаков
    batch_resp = requests.post(                      # POST /batch_predict
        f"{base_url}/batch_predict",
        json={"batch": batch_features}
    )
    print("\n=== /batch_predict ===")               # заголовок
    print(json.dumps(batch_resp.json(), indent=2))   # выводим JSON
except Exception as e:                               # если ошибка
    print(f"Ошибка /batch_predict: {e}")             # сообщение

---
## Шаг 5. Версионирование моделей

**Версионирование** - это систематическое отслеживание версий моделей, их метрик и метаданных.

### Зачем версионировать?

1. **Откат** - если новая модель хуже, вернуться к предыдущей
2. **A/B тестирование** - сравнивать версии на реальных данных
3. **Аудит** - знать, какая модель работала в какой момент
4. **Воспроизводимость** - точно воспроизвести любой эксперимент

In [ ]:
# ============================================================
# ШАГ 5: Реализуем простую систему версионирования моделей
# ============================================================

class ModelRegistry:                                 # реестр моделей
    def __init__(self, storage_dir='model_registry'):  # конструктор
        self.storage_dir = storage_dir                # директория хранения
        os.makedirs(storage_dir, exist_ok=True)      # создаём директорию
        self.models = {}                             # словарь зарегистрированных моделей
        self.metadata_file = os.path.join(storage_dir, 'metadata.json')  # файл метаданных
        self._load_metadata()                        # загружаем метаданные

    def _load_metadata(self):                        # загружаем метаданные из файла
        if os.path.exists(self.metadata_file):       # если файл существует
            with open(self.metadata_file, 'r') as f:  # открываем для чтения
                self.models = json.load(f)           # загружаем JSON
        else:                                        # иначе
            self.models = {}                         # пустой словарь

    def _save_metadata(self):                        # сохраняем метаданные в файл
        with open(self.metadata_file, 'w') as f:     # открываем для записи
            json.dump(self.models, f, indent=2, ensure_ascii=False)  # записываем JSON

    def register_model(self, version, model, metrics, description=""):  # регистрируем модель
        model_path = os.path.join(self.storage_dir, f'model_v{version}.pt')  # путь к файлу
        torch.save({                                 # сохраняем чекпоинт
            'model_state_dict': model.state_dict(),  # веса модели
            'version': version,                      # версия
            'metrics': metrics,                      # метрики
            'description': description,              # описание
            'registered_at': datetime.now().isoformat(),  # время регистрации
        }, model_path)                               # сохраняем в файл

        self.models[version] = {                     # записываем в реестр
            'path': model_path,                      # путь к файлу
            'metrics': metrics,                      # метрики
            'description': description,              # описание
            'registered_at': datetime.now().isoformat(),  # время
            'status': 'registered',                  # статус
        }
        self._save_metadata()                        # сохраняем метаданные
        print(f"Модель v{version} зарегистрирована!")  # подтверждение

    def load_model(self, version, model_class, **kwargs):  # загружаем модель по версии
        if version not in self.models:               # если версия не найдена
            raise ValueError(f"Версия {version} не найдена")  # ошибка
        model_path = self.models[version]['path']    # путь к файлу
        checkpoint = torch.load(model_path, map_location=device)  # загружаем
        model = model_class(**kwargs).to(device)     # создаём модель
        model.load_state_dict(checkpoint['model_state_dict'])  # загружаем веса
        model.eval()                                 # режим оценки
        return model                                 # возвращаем модель

    def promote(self, version, stage='production'):  # продвигаем модель в стадию
        if version not in self.models:               # если версия не найдена
            raise ValueError(f"Версия {version} не найдена")  # ошибка
        # Снимаем предыдущую модель с продакшена
        for v, data in self.models.items():          # перебираем все версии
            if data.get('status') == stage:          # если в той же стадии
                data['status'] = 'archived'          # архивируем
        self.models[version]['status'] = stage       # устанавливаем новую стадию
        self._save_metadata()                        # сохраняем
        print(f"Модель v{version} продвинута в {stage}!")  # подтверждение

    def list_models(self):                           # список всех моделей
        print(f"{'Версия':<10} {'Статус':<15} {'Точность':<12} {'Описание':<20}")  # заголовок
        print("-" * 60)                              # разделитель
        for version, data in sorted(self.models.items()):  # перебираем
            acc = data['metrics'].get('accuracy', 'N/A')  # точность
            print(f"v{version:<9} {data['status']:<15} {str(acc):<12} {data['description']:<20}")  # строка

    def get_production_model(self, model_class, **kwargs):  # получаем продакшен-модель
        for version, data in self.models.items():    # перебираем версии
            if data['status'] == 'production':       # если в продакшене
                return self.load_model(version, model_class, **kwargs)  # загружаем
        return None                                  # не найдено

# Создаём реестр и регистрируем модели
registry = ModelRegistry()                           # создаём реестр

# Регистрируем модель v1
registry.register_model(                             # регистрируем v1
    version='1.0.0',
    model=model_v1,
    metrics={'accuracy': accuracy, 'loss': loss.item()},
    description='SimpleClassifier 64-32'
)

# Регистрируем модель v2
registry.register_model(                             # регистрируем v2
    version='2.0.0',
    model=model_v2,
    metrics={'accuracy': accuracy2, 'loss': 0.0},
    description='ImprovedClassifier 128-64-32'
)

# Продвигаем v1 в продакшен
registry.promote('1.0.0', 'production')             # v1 -> production

# Показываем список моделей
print()
registry.list_models()                               # выводим список

In [ ]:
# ============================================================
# ШАГ 5 (продолжение): Визуализация версионирования
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(14, 6))      # создаём фигуру

# Данные для визуализации
versions = ['v1.0.0', 'v2.0.0']                     # версии
stages = ['production', 'registered']                # стадии
accs = [accuracy, accuracy2]                         # точности
colors_v = ['#4CAF50', '#2196F3']                    # цвета

# Рисуем этапы pipeline
stage_names = ['registered', 'staging', 'production']  # названия стадий
stage_colors = ['#E0E0E0', '#FFC107', '#4CAF50']    # цвета стадий
stage_x = [1, 4, 7]                                 # позиции x

for i, (name, color, x) in enumerate(zip(stage_names, stage_colors, stage_x)):  # рисуем стадии
    rect = plt.Rectangle((x - 0.8, 0.5), 1.6, 2,   # прямоугольник
                         facecolor=color, alpha=0.3, edgecolor='black', linewidth=2)  # стиль
    ax.add_patch(rect)                               # добавляем
    ax.text(x, 0.8, name, ha='center', fontsize=12, fontweight='bold')  # название

# Размещаем модели
model_positions = {'v1.0.0': (7, 1.5), 'v2.0.0': (1, 1.5)}  # позиции моделей
model_colors = {'v1.0.0': '#4CAF50', 'v2.0.0': '#2196F3'}    # цвета моделей

for ver, (x, y) in model_positions.items():         # для каждой модели
    idx = versions.index(ver)                        # индекс
    circle = plt.Circle((x, y), 0.35,               # круг
                        facecolor=model_colors[ver],  # цвет
                        edgecolor='black', linewidth=2,  # граница
                        alpha=0.8)                   # прозрачность
    ax.add_patch(circle)                             # добавляем
    ax.text(x, y, ver, ha='center', va='center',    # текст
            fontsize=9, fontweight='bold', color='white')  # стиль
    ax.text(x, y - 0.55, f'acc={accs[idx]:.3f}',    # точность
            ha='center', fontsize=9)                 # стиль

# Стрелки продвижения
ax.annotate('', xy=(6.2, 1.5), xytext=(1.8, 1.5),  # стрелка registered -> production
            arrowprops=dict(arrowstyle='->', lw=2, color='#333333', linestyle='--'))  # стиль
ax.text(4, 1.8, 'promote', ha='center', fontsize=10,  # текст
        fontstyle='italic', color='#333333')         # стиль

ax.set_xlim(-0.5, 9)                                # пределы x
ax.set_ylim(-0.5, 3)                                # пределы y
ax.set_title('Система версионирования моделей', fontsize=16, fontweight='bold')  # заголовок
ax.axis('off')                                       # скрываем оси

plt.tight_layout()                                    # подгоняем layout
plt.show()                                            # отображаем

---
## Шаг 6. A/B тестирование сервер

A/B тестирование позволяет сравнивать две модели на реальных данных, направляя часть трафика на каждую.

### Как работает A/B тестирование

```
Запрос -> [Маршрутизатор] -> Модель A (80% трафика)
                       |-> Модель B (20% трафика)
```

Мы создадим **второй FastAPI сервер** с:
- **/predict** - маршрутизация к модели A или B
- **/metrics** - статистика по каждой модели
- **/set_ratio** - изменение пропорции трафика
- **/compare** - сравнение моделей

In [ ]:
# ============================================================
# ШАГ 6: Создаём A/B тестирование сервер
# ============================================================

# Создаём второе FastAPI приложение для A/B тестирования
app_ab = FastAPI(                                    # A/B тестирование приложение
    title="A/B Testing API",                        # заголовок
    version="1.0.0",                                # версия
    description="A/B тестирование двух ML-моделей"  # описание
)

# Состояние A/B тестирования
ab_state = {                                         # состояние A/B теста
    'model_a': model_v1,                            # модель A (v1)
    'model_b': model_v2,                            # модель B (v2)
    'model_a_version': '1.0.0',                     # версия A
    'model_b_version': '2.0.0',                     # версия B
    'ratio_a': 0.7,                                 # доля трафика на модель A (70%)
    'predictions_a': 0,                             # количество предсказаний A
    'predictions_b': 0,                             # количество предсказаний B
    'correct_a': 0,                                 # правильные предсказания A
    'correct_b': 0,                                 # правильные предсказания B
    'latencies_a': [],                              # латентности A
    'latencies_b': [],                              # латентности B
    'start_time': time.time(),                      # время запуска
}

# Эндпоинт: предсказание с A/B маршрутизацией
@app_ab.post("/predict")                             # POST /predict
async def ab_predict(request: PredictionRequest):    # функция A/B предсказания
    start = time.time()                              # замеряем время

    # Определяем, к какой модели направить запрос
    rand_val = np.random.random()                    # случайное число [0, 1)
    use_model_a = rand_val < ab_state['ratio_a']    # маршрутизация

    # Выбираем модель
    model = ab_state['model_a'] if use_model_a else ab_state['model_b']  # выбираем модель
    version = ab_state['model_a_version'] if use_model_a else ab_state['model_b_version']  # версия
    model_label = 'A' if use_model_a else 'B'       # метка модели

    try:                                             # обрабатываем ошибки
        x = torch.tensor([request.features], dtype=torch.float32).to(device)  # тензор
        model.eval()                                 # режим оценки
        with torch.no_grad():                        # без градиентов
            logits = model(x)                        # прямой проход
            probs = F.softmax(logits, dim=1)         # вероятности
            pred_class = probs.argmax(dim=1).item()  # класс
            confidence = probs.max().item()           # уверенность

        latency = (time.time() - start) * 1000      # латентность

        # Обновляем статистику
        if use_model_a:                              # если модель A
            ab_state['predictions_a'] += 1           # увеличиваем счётчик A
            ab_state['latencies_a'].append(latency)  # сохраняем латентность A
        else:                                        # если модель B
            ab_state['predictions_b'] += 1           # увеличиваем счётчик B
            ab_state['latencies_b'].append(latency)  # сохраняем латентность B

        return {                                     # возвращаем результат
            'predicted_class': pred_class,           # класс
            'confidence': round(confidence, 4),      # уверенность
            'probabilities': [round(p, 4) for p in probs[0].cpu().tolist()],  # вероятности
            'model': model_label,                    # какая модель
            'model_version': version,                # версия модели
            'latency_ms': round(latency, 2),         # латентность
            'prediction_id': str(uuid.uuid4())[:8],  # ID предсказания
        }
    except Exception as e:                           # если ошибка
        raise HTTPException(status_code=500, detail=str(e))  # возвращаем 500

# Эндпоинт: метрики A/B тестирования
@app_ab.get("/metrics")                              # GET /metrics
async def ab_metrics():                              # функция метрик
    def calc_stats(latencies, count):                # вспомогательная функция
        if count == 0:                               # если нет данных
            return {'count': 0, 'avg_latency_ms': 0, 'p50_latency_ms': 0, 'p99_latency_ms': 0}
        return {                                     # вычисляем статистику
            'count': count,                          # количество
            'avg_latency_ms': round(np.mean(latencies), 2),  # средняя латентность
            'p50_latency_ms': round(np.percentile(latencies, 50), 2),  # медиана
            'p99_latency_ms': round(np.percentile(latencies, 99), 2),  # 99 перцентиль
        }

    return {                                         # возвращаем метрики
        'model_a': calc_stats(ab_state['latencies_a'], ab_state['predictions_a']),  # метрики A
        'model_b': calc_stats(ab_state['latencies_b'], ab_state['predictions_b']),  # метрики B
        'ratio_a': ab_state['ratio_a'],             # доля трафика A
        'ratio_b': round(1 - ab_state['ratio_a'], 2),  # доля трафика B
        'total_predictions': ab_state['predictions_a'] + ab_state['predictions_b'],  # всего
        'uptime_seconds': round(time.time() - ab_state['start_time'], 1),  # время работы
    }

# Эндпоинт: изменение пропорции трафика
@app_ab.post("/set_ratio")                           # POST /set_ratio
async def set_ratio(ratio_a: float = 0.7):           # функция установки пропорции
    if not 0 <= ratio_a <= 1:                        # проверяем диапазон
        raise HTTPException(status_code=400, detail="ratio_a должно быть в [0, 1]")  # ошибка
    ab_state['ratio_a'] = ratio_a                    # устанавливаем пропорцию
    return {                                         # возвращаем результат
        'ratio_a': ratio_a,                          # новая доля A
        'ratio_b': round(1 - ratio_a, 2),            # новая доля B
        'message': f'Трафик: {ratio_a*100:.0f}% -> A, {(1-ratio_a)*100:.0f}% -> B'  # сообщение
    }

# Эндпоинт: сравнение моделей
@app_ab.get("/compare")                              # GET /compare
async def compare_models():                           # функция сравнения
    # Тестируем обе модели на одних и тех же данных
    test_data = torch.randn(100, 20).to(device)      # 100 тестовых примеров

    results = {}                                     # результаты
    for label, model in [('A', ab_state['model_a']), ('B', ab_state['model_b'])]:  # для каждой модели
        model.eval()                                 # режим оценки
        with torch.no_grad():                        # без градиентов
            start = time.time()                      # замеряем время
            logits = model(test_data)                # прямой проход
            elapsed = (time.time() - start) * 1000   # время в мс
            probs = F.softmax(logits, dim=1)         # вероятности
            preds = probs.argmax(dim=1)              # классы
            avg_conf = probs.max(dim=1)[0].mean().item()  # средняя уверенность

        results[label] = {                           # сохраняем результаты
            'avg_confidence': round(avg_conf, 4),    # средняя уверенность
            'inference_time_ms': round(elapsed, 2),  # время inference
            'class_distribution': {str(c): (preds == c).sum().item() for c in range(3)},  # распределение
        }

    return results                                   # возвращаем результаты

print("A/B тестирование сервер создан!")             # подтверждение
print("Эндпоинты: /predict, /metrics, /set_ratio, /compare")  # список

In [ ]:
# ============================================================
# ШАГ 6 (продолжение): Запускаем A/B тестирование сервер с ngrok
# ============================================================

# Запускаем A/B сервер на порту 8001
def run_ab_server():                                 # функция запуска
    uvicorn.run(app_ab, host="0.0.0.0", port=8001, log_level="warning")  # uvicorn на 8001

ab_thread = threading.Thread(target=run_ab_server, daemon=True)  # поток
ab_thread.start()                                    # запускаем
time.sleep(2)                                        # ждём запуска

# Запускаем ngrok для A/B сервера
try:                                                 # пытаемся
    ab_public_url = ngrok.connect(8001)              # туннель на 8001
    print(f"A/B тестирование сервер запущен!")        # подтверждение
    print(f"Публичный URL: {ab_public_url}")         # публичный URL
    print(f"Документация: {ab_public_url}/docs")     # Swagger UI
except Exception as e:                               # если ошибка
    print(f"ngrok не запущен: {e}")                  # сообщение
    print("A/B сервер работает локально на http://localhost:8001")  # локальный доступ

In [ ]:
# ============================================================
# ШАГ 6 (продолжение): Тестируем A/B тестирование сервер
# ============================================================

ab_base_url = "http://localhost:8001"                # базовый URL A/B сервера

# Тест 1: Устанавливаем пропорцию 50/50
try:                                                 # обрабатываем ошибки
    ratio_resp = requests.post(f"{ab_base_url}/set_ratio?ratio_a=0.5")  # 50/50
    print("=== /set_ratio (50/50) ===")              # заголовок
    print(json.dumps(ratio_resp.json(), indent=2))   # выводим JSON
except Exception as e:                               # если ошибка
    print(f"Ошибка /set_ratio: {e}")                 # сообщение

# Тест 2: Отправляем 20 запросов и смотрим распределение
model_a_count = 0                                    # счётчик модели A
model_b_count = 0                                    # счётчик модели B
for i in range(20):                                  # 20 запросов
    try:                                             # обрабатываем ошибки
        features = np.random.randn(20).tolist()     # случайные признаки
        resp = requests.post(                        # POST /predict
            f"{ab_base_url}/predict",
            json={"features": features}
        )
        result = resp.json()                         # парсим JSON
        if result.get('model') == 'A':              # если модель A
            model_a_count += 1                       # увеличиваем счётчик A
        else:                                        # если модель B
            model_b_count += 1                       # увеличиваем счётчик B
    except Exception as e:                           # если ошибка
        pass                                         # пропускаем

print(f"\n=== 20 запросов (50/50) ===")             # заголовок
print(f"Модель A: {model_a_count} запросов")        # результат A
print(f"Модель B: {model_b_count} запросов")        # результат B

# Тест 3: Метрики
try:                                                 # обрабатываем ошибки
    metrics_resp = requests.get(f"{ab_base_url}/metrics")  # GET /metrics
    print("\n=== /metrics ===")                     # заголовок
    print(json.dumps(metrics_resp.json(), indent=2))  # выводим JSON
except Exception as e:                               # если ошибка
    print(f"Ошибка /metrics: {e}")                   # сообщение

# Тест 4: Сравнение моделей
try:                                                 # обрабатываем ошибки
    compare_resp = requests.get(f"{ab_base_url}/compare")  # GET /compare
    print("\n=== /compare ===")                     # заголовок
    print(json.dumps(compare_resp.json(), indent=2))  # выводим JSON
except Exception as e:                               # если ошибка
    print(f"Ошибка /compare: {e}")                   # сообщение

---
## Шаг 7. Мониторинг-дашборд

Мониторинг - ключевой компонент MLOps. Без него вы не узнаете, что модель деградирует, пока не станет слишком поздно.

### Ключевые метрики мониторинга

| Метрика | Что показывает | Порог тревоги |
|---------|---------------|---------------|
| **Латентность** | Время ответа API | p99 > 500ms |
| **Error rate** | Доля ошибочных запросов | > 1% |
| **Distribution** | Распределение предсказаний | Смещение > 20% |
| **Confidence** | Средняя уверенность модели | < 0.6 |

In [ ]:
# ============================================================
# ШАГ 7: Собираем мониторинг-метрики из продакшен-сервера
# ============================================================

# Генерируем трафик для сбора метрик
monitoring_data = {                                  # данные мониторинга
    'latencies': [],                                 # латентности
    'predictions': [],                               # предсказания
    'confidences': [],                               # уверенности
    'errors': 0,                                     # количество ошибок
    'timestamps': [],                                # временные метки
}

# Отправляем 100 запросов на продакшен-сервер
for i in range(100):                                 # 100 запросов
    try:                                             # обрабатываем ошибки
        features = np.random.randn(20).tolist()     # случайные признаки
        start = time.time()                          # замеряем время
        resp = requests.post(                        # POST /predict
            f"{base_url}/predict",
            json={"features": features}
        )
        latency = (time.time() - start) * 1000      # латентность в мс
        result = resp.json()                         # парсим JSON

        monitoring_data['latencies'].append(latency)  # сохраняем латентность
        monitoring_data['predictions'].append(result['predicted_class'])  # предсказание
        monitoring_data['confidences'].append(result['confidence'])  # уверенность
        monitoring_data['timestamps'].append(time.time())  # временная метка
    except Exception as e:                           # если ошибка
        monitoring_data['errors'] += 1               # увеличиваем счётчик ошибок

print(f"Собрано данных: {len(monitoring_data['latencies'])} запросов")  # результат
print(f"Ошибок: {monitoring_data['errors']}")        # количество ошибок
print(f"Средняя латентность: {np.mean(monitoring_data['latencies']):.1f} ms")  # средняя
print(f"p99 латентность: {np.percentile(monitoring_data['latencies'], 99):.1f} ms")  # p99

In [ ]:
# ============================================================
# ШАГ 7 (продолжение): Визуализируем мониторинг-дашборд
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))    # 4 подграфика

# --- 1. Латентность во времени ---
ax = axes[0, 0]                                     # первый подграфик
latencies = monitoring_data['latencies']             # латентности
ax.plot(latencies, color='#2196F3', alpha=0.5, linewidth=0.8)  # линия
# Скользящее среднее
window = 10                                          # размер окна
if len(latencies) >= window:                         # если достаточно данных
    moving_avg = np.convolve(latencies, np.ones(window)/window, mode='valid')  # скользящее среднее
    ax.plot(range(window-1, len(latencies)), moving_avg, color='#F44336', linewidth=2, label='Скользящее среднее')  # рисуем
ax.axhline(y=np.percentile(latencies, 95), color='#FF9800', linestyle='--', label='p95')  # порог p95
ax.set_xlabel('Номер запроса', fontsize=11)         # подпись оси
ax.set_ylabel('Латентность (мс)', fontsize=11)      # подпись оси
ax.set_title('Латентность API во времени', fontsize=13, fontweight='bold')  # заголовок
ax.legend(fontsize=10)                              # легенда

# --- 2. Распределение латентности ---
ax = axes[0, 1]                                     # второй подграфик
ax.hist(latencies, bins=30, color='#2196F3', edgecolor='black', alpha=0.7)  # гистограмма
ax.axvline(x=np.mean(latencies), color='#4CAF50', linestyle='--', linewidth=2, label=f'Среднее: {np.mean(latencies):.1f}ms')  # среднее
ax.axvline(x=np.percentile(latencies, 99), color='#F44336', linestyle='--', linewidth=2, label=f'p99: {np.percentile(latencies, 99):.1f}ms')  # p99
ax.set_xlabel('Латентность (мс)', fontsize=11)      # подпись оси
ax.set_ylabel('Количество', fontsize=11)            # подпись оси
ax.set_title('Распределение латентности', fontsize=13, fontweight='bold')  # заголовок
ax.legend(fontsize=10)                              # легенда

# --- 3. Распределение предсказаний ---
ax = axes[1, 0]                                     # третий подграфик
preds = monitoring_data['predictions']               # предсказания
class_counts = {c: preds.count(c) for c in set(preds)}  # подсчёт классов
classes = sorted(class_counts.keys())                # сортировка
counts = [class_counts[c] for c in classes]          # количества
colors_pred = ['#4CAF50', '#2196F3', '#FF9800']     # цвета
bars = ax.bar([f'Класс {c}' for c in classes], counts, color=colors_pred[:len(classes)], edgecolor='black')  # столбцы
for bar, count in zip(bars, counts):                 # добавляем значения
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(count), ha='center', fontweight='bold')  # текст
ax.set_ylabel('Количество', fontsize=11)            # подпись оси
ax.set_title('Распределение предсказаний по классам', fontsize=13, fontweight='bold')  # заголовок

# --- 4. Уверенность модели ---
ax = axes[1, 1]                                     # четвёртый подграфик
confidences = monitoring_data['confidences']         # уверенности
ax.plot(confidences, color='#9C27B0', alpha=0.5, linewidth=0.8)  # линия
if len(confidences) >= window:                       # если достаточно данных
    moving_conf = np.convolve(confidences, np.ones(window)/window, mode='valid')  # скользящее среднее
    ax.plot(range(window-1, len(confidences)), moving_conf, color='#F44336', linewidth=2, label='Скользящее среднее')  # рисуем
ax.axhline(y=0.6, color='#FF9800', linestyle='--', label='Порог тревоги (0.6)')  # порог
ax.set_xlabel('Номер запроса', fontsize=11)         # подпись оси
ax.set_ylabel('Уверенность', fontsize=11)           # подпись оси
ax.set_title('Уверенность модели во времени', fontsize=13, fontweight='bold')  # заголовок
ax.legend(fontsize=10)                              # легенда

plt.suptitle('Мониторинг-дашборд продакшен API', fontsize=16, fontweight='bold')  # общий заголовок
plt.tight_layout()                                    # подгоняем layout
plt.show()                                            # отображаем

---
## Шаг 8. Интерактивный деплой-эксплорер

Используем **ipywidgets** для интерактивного исследования параметров деплоя.

### Настраиваемые параметры
- **batch_size** - размер батча для inference
- **workers** - количество воркеров
- **timeout** - таймаут запроса
- **model_version** - версия модели для деплоя

In [ ]:
# ============================================================
# ШАГ 8: Интерактивный деплой-эксплорер с ipywidgets
# ============================================================

# Функция для симуляции деплоя с разными параметрами
def simulate_deployment(batch_size, workers, timeout_ms, model_version):  # симуляция деплоя
    # Симулируем время inference в зависимости от параметров
    base_latency = 5.0                              # базовая латентность (мс)
    # Большой batch_size увеличивает время, но улучшает throughput
    batch_factor = np.log2(batch_size + 1)          # фактор батча
    # Больше воркеров - меньше время (до предела)
    worker_factor = 1.0 / (1 + 0.3 * workers)      # фактор воркеров

    # Выбираем модель по версии
    if model_version == 'v1.0.0':                   # если v1
        model = model_v1                            # используем v1
        model_name = "SimpleClassifier (64-32)"     # имя
    else:                                           # если v2
        model = model_v2                            # используем v2
        model_name = "ImprovedClassifier (128-64-32)"  # имя

    # Симулируем inference
    test_batch = torch.randn(batch_size, 20).to(device)  # тестовый батч
    model.eval()                                     # режим оценки

    latencies = []                                   # список латентностей
    successful = 0                                   # успешные запросы
    timeouts = 0                                     # таймауты

    for _ in range(10):                              # 10 запросов
        start = time.time()                          # замеряем время
        with torch.no_grad():                        # без градиентов
            _ = model(test_batch)                    # прямой проход
        elapsed = (time.time() - start) * 1000      # время в мс
        # Добавляем случайную вариацию (сеть, очередь)
        total_latency = elapsed * batch_factor * worker_factor + np.random.exponential(2)  # общая латентность
        latencies.append(total_latency)              # сохраняем

        if total_latency > timeout_ms:               # если превышен таймаут
            timeouts += 1                            # увеличиваем счётчик
        else:                                        # иначе
            successful += 1                          # увеличиваем успешные

    # Вычисляем метрики
    avg_latency = np.mean(latencies)                 # средняя латентность
    p99_latency = np.percentile(latencies, 99)       # p99
    throughput = successful / (sum(latencies) / 1000) if sum(latencies) > 0 else 0  # throughput (req/s)
    error_rate = timeouts / 10 * 100                 # процент таймаутов

    # Визуализация
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))  # 3 подграфика

    # 1. Латентность
    ax = axes[0]                                     # первый
    ax.bar(['Средняя', 'p99', 'p50'],               # метрики
           [avg_latency, p99_latency, np.median(latencies)],  # значения
           color=['#4CAF50', '#F44336', '#2196F3'], edgecolor='black')  # цвета
    ax.axhline(y=timeout_ms, color='black', linestyle='--', label=f'Таймаут: {timeout_ms}ms')  # порог
    ax.set_ylabel('Латентность (мс)', fontsize=11)   # подпись
    ax.set_title('Латентность', fontsize=13, fontweight='bold')  # заголовок
    ax.legend(fontsize=9)                            # легенда

    # 2. Throughput
    ax = axes[1]                                     # второй
    ax.bar(['Throughput\n(req/s)', 'Error rate\n(%)'],  # метрики
           [throughput, error_rate],                  # значения
           color=['#4CAF50', '#F44336'], edgecolor='black')  # цвета
    ax.set_title('Производительность', fontsize=13, fontweight='bold')  # заголовок

    # 3. Информационная панель
    ax = axes[2]                                     # третий
    ax.axis('off')                                   # скрываем оси
    info_text = (                                    # информационный текст
        f"Конфигурация деплоя\n"
        f"{'='*30}\n"
        f"Модель: {model_name}\n"
        f"Версия: {model_version}\n"
        f"Batch size: {batch_size}\n"
        f"Workers: {workers}\n"
        f"Таймаут: {timeout_ms} ms\n"
        f"{'='*30}\n"
        f"Результаты\n"
        f"{'='*30}\n"
        f"Средняя латентность: {avg_latency:.1f} ms\n"
        f"p99 латентность: {p99_latency:.1f} ms\n"
        f"Throughput: {throughput:.1f} req/s\n"
        f"Таймауты: {timeouts}/10 ({error_rate:.0f}%)\n"
        f"Статус: {'OK' if error_rate < 10 else 'WARNING' if error_rate < 30 else 'CRITICAL'}"
    )
    ax.text(0.1, 0.5, info_text, fontsize=11,       # выводим текст
            family='monospace', transform=ax.transAxes,  # моноширинный шрифт
            verticalalignment='center',               # выравнивание
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))  # рамка

    plt.suptitle('Интерактивный деплой-эксплорер', fontsize=15, fontweight='bold')  # заголовок
    plt.tight_layout()                                # подгоняем layout
    plt.show()                                        # отображаем

# Создаём интерактивные виджеты
batch_size_widget = widgets.IntSlider(               # виджет batch_size
    value=8, min=1, max=64, step=1,                 # параметры
    description='Batch size:',                       # подпись
    style={'description_width': 'initial'},          # стиль
    layout=Layout(width='400px')                     # размер
)

workers_widget = widgets.IntSlider(                  # виджет workers
    value=4, min=1, max=16, step=1,                 # параметры
    description='Workers:',                          # подпись
    style={'description_width': 'initial'},          # стиль
    layout=Layout(width='400px')                     # размер
)

timeout_widget = widgets.IntSlider(                  # виджет timeout
    value=100, min=10, max=500, step=10,            # параметры
    description='Timeout (ms):',                     # подпись
    style={'description_width': 'initial'},          # стиль
    layout=Layout(width='400px')                     # размер
)

model_widget = widgets.Dropdown(                     # виджет model_version
    options=['v1.0.0', 'v2.0.0'],                   # варианты
    value='v1.0.0',                                 # значение по умолчанию
    description='Model:',                            # подпись
    style={'description_width': 'initial'},          # стиль
    layout=Layout(width='400px')                     # размер
)

# Создаём интерактивный интерфейс
interactive_deploy = interactive(                    # интерактивный виджет
    simulate_deployment,                             # функция
    batch_size=batch_size_widget,                    # виджет batch_size
    workers=workers_widget,                          # виджет workers
    timeout_ms=timeout_widget,                       # виджет timeout
    model_version=model_widget                       # виджет модели
)

print("Интерактивный деплой-эксплорер создан!")      # подтверждение
display(interactive_deploy)                          # отображаем

---
## Шаг 9. Детекция дрифта данных

**Data drift** (дрейф данных) - изменение распределения входных данных со временем. Это главная причина деградации моделей в продакшене.

### Типы дрифта

| Тип | Описание | Пример |
|-----|----------|--------|
| **Covariate shift** | Меняется P(X) | Пользователи стали старше |
| **Concept drift** | Меняется P(Y\|X) | Спам-фильтр устарел |
| **Prior shift** | Меняется P(Y) | Баланс классов изменился |

### Статистические тесты для детекции

| Тест | Для чего | Когда использовать |
|------|----------|--------------------|
| **KS test** | Непрерывные данные | Сравнение распределений |
| **Chi-squared** | Категориальные данные | Сравнение частот |
| **PSI** | Любые данные | Population Stability Index |

In [ ]:
# ============================================================
# ШАГ 9: Реализуем детекцию дрифта данных
# ============================================================

class DriftDetector:                                 # детектор дрифта
    def __init__(self, reference_data):              # конструктор
        # Сохраняем эталонные данные
        self.reference = reference_data.cpu().numpy() if torch.is_tensor(reference_data) else reference_data  # конвертируем
        self.n_features = self.reference.shape[1]    # количество признаков
        # Вычисляем эталонную статистику
        self.ref_means = np.mean(self.reference, axis=0)  # средние
        self.ref_stds = np.std(self.reference, axis=0)    # стандартные отклонения

    def ks_test(self, new_data, alpha=0.05):         # тест Колмогорова-Смирнова
        new = new_data.cpu().numpy() if torch.is_tensor(new_data) else new_data  # конвертируем
        results = {}                                 # результаты
        drift_detected = False                       # флаг дрифта

        for i in range(self.n_features):             # для каждого признака
            stat, p_value = stats.ks_2samp(          # KS тест
                self.reference[:, i],                # эталонное распределение
                new[:, i]                             # новое распределение
            )
            is_drift = p_value < alpha               # дрифт если p < alpha
            results[f'feature_{i}'] = {              # сохраняем результат
                'ks_statistic': round(stat, 4),      # статистика
                'p_value': round(p_value, 6),        # p-value
                'is_drift': is_drift,                # есть ли дрифт
            }
            if is_drift:                             # если обнаружен дрифт
                drift_detected = True                # устанавливаем флаг

        return results, drift_detected               # возвращаем результаты

    def psi_score(self, new_data, n_bins=10):        # Population Stability Index
        new = new_data.cpu().numpy() if torch.is_tensor(new_data) else new_data  # конвертируем
        psi_values = []                              # список PSI

        for i in range(self.n_features):             # для каждого признака
            # Определяем границы бинов по эталонным данным
            _, bin_edges = np.histogram(self.reference[:, i], bins=n_bins)  # границы бинов

            # Считаем частоты в бинах
            ref_counts, _ = np.histogram(self.reference[:, i], bins=bin_edges)  # эталон
            new_counts, _ = np.histogram(new[:, i], bins=bin_edges)  # новые

            # Нормализуем в вероятности
            ref_probs = ref_counts / ref_counts.sum() + 1e-6  # эталонные вероятности
            new_probs = new_counts / new_counts.sum() + 1e-6  # новые вероятности

            # Вычисляем PSI
            psi = np.sum((new_probs - ref_probs) * np.log(new_probs / ref_probs))  # PSI
            psi_values.append(psi)                   # сохраняем

        avg_psi = np.mean(psi_values)                # средний PSI
        # PSI < 0.1 - нет дрифта, 0.1-0.2 - умеренный, > 0.2 - значительный
        if avg_psi < 0.1:                            # если маленький
            level = "Нет дрифта"                     # нет дрифта
        elif avg_psi < 0.2:                          # если средний
            level = "Умеренный дрифт"                # умеренный
        else:                                        # если большой
            level = "Значительный дрифт"             # значительный

        return avg_psi, level, psi_values            # возвращаем результаты

    def feature_drift_summary(self, new_data):       # сводка по признакам
        new = new_data.cpu().numpy() if torch.is_tensor(new_data) else new_data  # конвертируем
        summary = []                                 # сводка

        for i in range(self.n_features):             # для каждого признака
            ref_mean = self.ref_means[i]             # эталонное среднее
            new_mean = np.mean(new[:, i])            # новое среднее
            ref_std = self.ref_stds[i]               # эталонное отклонение
            new_std = np.std(new[:, i])              # новое отклонение
            shift = abs(new_mean - ref_mean) / (ref_std + 1e-8)  # величина сдвига

            summary.append({                         # добавляем в сводку
                'feature': i,                        # номер признака
                'ref_mean': round(ref_mean, 4),      # эталонное среднее
                'new_mean': round(new_mean, 4),      # новое среднее
                'shift_sigma': round(shift, 4),      # сдвиг в сигмах
            })

        return summary                               # возвращаем сводку

# Создаём эталонные данные (обучающая выборка)
reference_data = X_train.cpu() if X_train.is_cuda else X_train  # эталонные данные
detector = DriftDetector(reference_data)             # создаём детектор

print("Детектор дрифта создан!")                     # подтверждение
print(f"Эталонная выборка: {reference_data.shape[0]} примеров, {reference_data.shape[1]} признаков")  # информация

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Тестируем детекцию дрифта на разных сценариях
# ============================================================

# Сценарий 1: Нет дрифта (данные из того же распределения)
no_drift_data = torch.randn(200, 20).to(device)     # данные без дрифта
ks_results_no, ks_drift_no = detector.ks_test(no_drift_data)  # KS тест
psi_no, psi_level_no, _ = detector.psi_score(no_drift_data)  # PSI

print("=== Сценарий 1: Нет дрифта ===")              # заголовок
print(f"KS тест: дрифт обнаружен = {ks_drift_no}")  # результат KS
print(f"PSI: {psi_no:.4f} ({psi_level_no})")        # результат PSI

# Сценарий 2: Умеренный дрифт (сдвиг среднего на 0.5)
moderate_drift_data = torch.randn(200, 20).to(device) + 0.5  # сдвиг на 0.5
ks_results_mod, ks_drift_mod = detector.ks_test(moderate_drift_data)  # KS тест
psi_mod, psi_level_mod, _ = detector.psi_score(moderate_drift_data)  # PSI

print(f"\n=== Сценарий 2: Умеренный дрифт (сдвиг +0.5) ===")  # заголовок
print(f"KS тест: дрифт обнаружен = {ks_drift_mod}")  # результат KS
print(f"PSI: {psi_mod:.4f} ({psi_level_mod})")      # результат PSI

# Сценарий 3: Значительный дрифт (сдвиг среднего на 2.0)
heavy_drift_data = torch.randn(200, 20).to(device) + 2.0  # сдвиг на 2.0
ks_results_heavy, ks_drift_heavy = detector.ks_test(heavy_drift_data)  # KS тест
psi_heavy, psi_level_heavy, psi_features = detector.psi_score(heavy_drift_data)  # PSI

print(f"\n=== Сценарий 3: Значительный дрифт (сдвиг +2.0) ===")  # заголовок
print(f"KS тест: дрифт обнаружен = {ks_drift_heavy}")  # результат KS
print(f"PSI: {psi_heavy:.4f} ({psi_level_heavy})")  # результат PSI

# Сценарий 4: Изменение дисперсии
var_drift_data = torch.randn(200, 20).to(device) * 3.0  # увеличиваем дисперсию в 3 раза
ks_results_var, ks_drift_var = detector.ks_test(var_drift_data)  # KS тест
psi_var, psi_level_var, _ = detector.psi_score(var_drift_data)  # PSI

print(f"\n=== Сценарий 4: Изменение дисперсии (x3) ===")  # заголовок
print(f"KS тест: дрифт обнаружен = {ks_drift_var}")  # результат KS
print(f"PSI: {psi_var:.4f} ({psi_level_var})")      # результат PSI

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Визуализация детекции дрифта
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))    # 4 подграфика

# Выбираем 2 признака для визуализации
features_to_show = [0, 5]                            # признаки 0 и 5
colors_ref = '#2196F3'                               # цвет эталона
colors_new = '#F44336'                               # цвет новых данных

scenarios = [                                         # сценарии
    ('Нет дрифта', no_drift_data.cpu().numpy()),     # сц. 1
    ('Умеренный дрифт (+0.5)', moderate_drift_data.cpu().numpy()),  # сц. 2
    ('Значительный дрифт (+2.0)', heavy_drift_data.cpu().numpy()),  # сц. 3
    ('Изменение дисперсии (x3)', var_drift_data.cpu().numpy()),     # сц. 4
]

for idx, (title, data) in enumerate(scenarios):      # для каждого сценария
    ax = axes[idx // 2, idx % 2]                     # выбираем подграфик
    feat = features_to_show[0]                       # признак для визуализации

    # Рисуем распределения
    ax.hist(reference_data[:, feat].numpy(), bins=30, alpha=0.5,  # эталон
            color=colors_ref, label='Эталон', edgecolor='black', density=True)  # стиль
    ax.hist(data[:, feat], bins=30, alpha=0.5,       # новые данные
            color=colors_new, label='Новые', edgecolor='black', density=True)  # стиль

    ax.set_title(title, fontsize=13, fontweight='bold')  # заголовок
    ax.legend(fontsize=10)                            # легенда
    ax.set_xlabel(f'Признак {feat}', fontsize=11)    # подпись оси
    ax.set_ylabel('Плотность', fontsize=11)          # подпись оси

plt.suptitle('Детекция дрифта данных: сравнение распределений', fontsize=16, fontweight='bold')  # общий заголовок
plt.tight_layout()                                    # подгоняем layout
plt.show()                                            # отображаем

---
## Шаг 10. Автоматическое переобучение

Когда дрифт обнаружен, система должна автоматически:
1. Обнаружить дрифт -> подать сигнал тревоги
2. Собрать новые данные для переобучения
3. Переобучить модель на новых данных
4. Оценить качество новой модели
5. Если лучше -> задеплоить, иначе -> откатиться

```
Дрифт обнаружен -> Сбор данных -> Переобучение -> Оценка -> Деплой
      ^                                                         |
      |_________________________________________________________|
```

In [ ]:
# ============================================================
# ШАГ 10: Реализуем автоматическое переобучение при дрифте
# ============================================================

class AutoRetrainPipeline:                           # пайплайн авто-переобучения
    def __init__(self, model_class, detector, registry):  # конструктор
        self.model_class = model_class               # класс модели
        self.detector = detector                     # детектор дрифта
        self.registry = registry                     # реестр моделей
        self.retrain_history = []                    # история переобучений
        self.drift_threshold_psi = 0.15              # порог PSI для дрифта
        self.min_accuracy_improvement = 0.01         # минимальное улучшение точности

    def check_drift(self, new_data):                 # проверяем дрифт
        psi, level, psi_features = self.detector.psi_score(new_data)  # PSI
        ks_results, ks_drift = self.detector.ks_test(new_data)  # KS тест
        is_drift = psi > self.drift_threshold_psi or ks_drift  # дрифт?

        return {                                     # возвращаем результат
            'is_drift': is_drift,                    # есть ли дрифт
            'psi': psi,                              # PSI значение
            'psi_level': level,                      # уровень PSI
            'ks_drift': ks_drift,                    # результат KS теста
        }

    def retrain(self, new_data, new_labels, current_version):  # переобучаем модель
        print(f"\n{'='*50}")                        # разделитель
        print(f"ПЕРЕОБУЧЕНИЕ: drift detected!")      # сообщение
        print(f"{'='*50}")                           # разделитель

        # Шаг 1: Создаём новую версию модели
        new_version = self._increment_version(current_version)  # увеличиваем версию
        print(f"Новая версия: {new_version}")        # выводим версию

        # Шаг 2: Обучаем новую модель
        new_model = self.model_class().to(device)    # создаём модель
        optimizer_new = optim.Adam(new_model.parameters(), lr=0.01)  # оптимизатор
        criterion = nn.CrossEntropyLoss()            # функция потерь

        new_model.train()                            # режим обучения
        for epoch in range(50):                      # 50 эпох
            optimizer_new.zero_grad()                # обнуляем градиенты
            outputs = new_model(new_data)            # прямой проход
            loss = criterion(outputs, new_labels)    # loss
            loss.backward()                          # обратное распространение
            optimizer_new.step()                     # шаг оптимизации

        # Шаг 3: Оцениваем качество
        new_model.eval()                             # режим оценки
        with torch.no_grad():                        # без градиентов
            outputs = new_model(new_data)            # прямой проход
            preds = outputs.argmax(dim=1)            # предсказания
            new_accuracy = (preds == new_labels).float().mean().item()  # точность

        print(f"Точность новой модели: {new_accuracy:.4f}")  # результат

        # Шаг 4: Регистрируем модель
        self.registry.register_model(                # регистрируем
            version=new_version,
            model=new_model,
            metrics={'accuracy': new_accuracy, 'loss': loss.item()},
            description=f'Auto-retrained on drifted data'
        )

        # Шаг 5: Записываем в историю
        retrain_record = {                           # запись
            'timestamp': datetime.now().isoformat(),  # время
            'old_version': current_version,           # старая версия
            'new_version': new_version,               # новая версия
            'new_accuracy': new_accuracy,             # новая точность
            'psi_before': self.detector.psi_score(new_data)[0],  # PSI до
        }
        self.retrain_history.append(retrain_record)  # добавляем в историю

        return new_model, new_version, new_accuracy   # возвращаем результаты

    def _increment_version(self, version):           # увеличиваем версию
        parts = version.split('.')                   # разделяем по точкам
        patch = int(parts[-1]) + 1                   # увеличиваем patch
        return '.'.join(parts[:-1] + [str(patch)])   # собираем обратно

# Создаём пайплайн авто-переобучения
auto_pipeline = AutoRetrainPipeline(                 # пайплайн
    model_class=lambda: SimpleClassifier(input_dim=20, hidden_dim=64, num_classes=3),  # класс модели
    detector=detector,                               # детектор дрифта
    registry=registry                                # реестр моделей
)

print("Пайплайн авто-переобучения создан!")           # подтверждение

In [ ]:
# ============================================================
# ШАГ 10 (продолжение): Запускаем автоматическое переобучение
# ============================================================

# Симулируем поступление данных с дрифтом
print("Мониторинг входящих данных...")               # заголовок

# Данные с дрифтом (поступают в продакшен)
drifted_data = torch.randn(200, 20).to(device) + 1.5  # данные со сдвигом
drifted_labels = torch.randint(0, 3, (200,)).to(device)  # метки

# Шаг 1: Проверяем дрифт
drift_result = auto_pipeline.check_drift(drifted_data)  # проверяем
print(f"\nПроверка дрифта:")                        # заголовок
print(f"  PSI: {drift_result['psi']:.4f} ({drift_result['psi_level']})")  # PSI
print(f"  KS тест: дрифт = {drift_result['ks_drift']}")  # KS
print(f"  Общий дрифт: {drift_result['is_drift']}")  # результат

# Шаг 2: Если дрифт обнаружен - переобучаем
if drift_result['is_drift']:                         # если есть дрифт
    print("\nДрифт обнаружен! Запускаем переобучение...")  # сообщение
    new_model, new_version, new_accuracy = auto_pipeline.retrain(  # переобучаем
        drifted_data, drifted_labels, '2.0.0'       # данные, метки, текущая версия
    )
    print(f"\nПереобучение завершено!")             # подтверждение
    print(f"Новая версия: {new_version}")            # версия
    print(f"Точность: {new_accuracy:.4f}")           # точность

    # Шаг 3: Продвигаем новую модель в продакшен
    if new_accuracy > 0.3:                           # если точность приемлемая
        registry.promote(new_version, 'production')  # продвигаем
        print(f"Модель v{new_version} задеплоена в продакшен!")  # подтверждение

# Показываем обновлённый реестр
print("\nОбновлённый реестр моделей:")              # заголовок
registry.list_models()                                # список моделей

In [ ]:
# ============================================================
# ШАГ 10 (продолжение): Визуализация пайплайна авто-переобучения
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))    # 2 подграфика

# --- Левый: Схема пайплайна ---
ax = axes[0]                                         # первый подграфик
ax.set_xlim(0, 10)                                   # пределы x
ax.set_ylim(0, 10)                                   # пределы y

# Этапы пайплайна
pipeline_steps = [                                    # этапы
    ('Мониторинг\nданных', 2, 8, '#2196F3'),        # шаг 1
    ('Детекция\nдрифта', 2, 6, '#FF9800'),          # шаг 2
    ('Пере-\nобучение', 2, 4, '#4CAF50'),           # шаг 3
    ('Оценка\nкачества', 2, 2, '#9C27B0'),          # шаг 4
    ('Деплой', 8, 5, '#F44336'),                     # шаг 5
]

for text, x, y, color in pipeline_steps:             # рисуем этапы
    rect = plt.Rectangle((x - 1.2, y - 0.6), 2.4, 1.2,  # прямоугольник
                         facecolor=color, alpha=0.7, edgecolor='black', linewidth=2)  # стиль
    ax.add_patch(rect)                                # добавляем
    ax.text(x, y, text, ha='center', va='center',   # текст
            fontsize=10, fontweight='bold', color='white')  # стиль

# Стрелки между этапами
arrows = [(2, 7.4, 2, 6.6), (2, 5.4, 2, 4.6), (2, 3.4, 2, 2.6),  # вниз
          (3.2, 2, 6.8, 5)]                          # к деплою
for x1, y1, x2, y2 in arrows:                       # рисуем стрелки
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),   # стрелка
                arrowprops=dict(arrowstyle='->', lw=2, color='#333333'))  # стиль

# Обратная связь (деплой -> мониторинг)
ax.annotate('', xy=(3.2, 8), xytext=(6.8, 5.6),    # стрелка обратной связи
            arrowprops=dict(arrowstyle='->', lw=2, color='#F44336',  # стиль
                           connectionstyle='arc3,rad=-0.3', linestyle='--'))  # кривая
ax.text(6, 7.5, 'обратная\nсвязь', fontsize=9,    # текст
        ha='center', color='#F44336', fontstyle='italic')  # стиль

ax.set_title('Пайплайн авто-переобучения', fontsize=14, fontweight='bold')  # заголовок
ax.axis('off')                                       # скрываем оси

# --- Правый: История переобучений ---
ax = axes[1]                                         # второй подграфик
if auto_pipeline.retrain_history:                    # если есть история
    versions = [r['old_version'] + ' -> ' + r['new_version'] for r in auto_pipeline.retrain_history]  # версии
    accuracies = [r['new_accuracy'] for r in auto_pipeline.retrain_history]  # точности
    ax.barh(versions, accuracies, color='#4CAF50', edgecolor='black')  # столбцы
    for i, acc in enumerate(accuracies):             # добавляем значения
        ax.text(acc + 0.01, i, f'{acc:.4f}', va='center', fontsize=11, fontweight='bold')  # текст
    ax.set_xlabel('Точность', fontsize=12)           # подпись оси
    ax.set_title('История переобучений', fontsize=14, fontweight='bold')  # заголовок
else:                                                # если нет истории
    ax.text(0.5, 0.5, 'Переобучений ещё не было',   # сообщение
            ha='center', va='center', fontsize=14, transform=ax.transAxes)  # стиль

plt.tight_layout()                                    # подгоняем layout
plt.show()                                            # отображаем

---
## Шаг 11. Нагрузочное тестирование

Нагрузочное тестирование показывает, как сервер справляется с большим количеством запросов.

### Метрики нагрузочного тестирования

| Метрика | Описание | Цель |
|---------|----------|------|
| **RPS** | Запросов в секунду | Максимальное значение |
| **Latency p50** | Медианная латентность | < 50ms |
| **Latency p99** | 99-й перцентиль | < 200ms |
| **Error rate** | Процент ошибок | < 0.1% |
| **Throughput** | Пропускная способность | Зависит от задачи |

In [ ]:
# ============================================================
# ШАГ 11: Нагрузочное тестирование продакшен-сервера
# ============================================================

class LoadTester:                                    # нагрузочный тестер
    def __init__(self, base_url):                    # конструктор
        self.base_url = base_url                     # базовый URL
        self.results = []                            # результаты тестов

    def run_test(self, n_requests=100, n_concurrent=1):  # запуск теста
        print(f"Нагрузочный тест: {n_requests} запросов, {n_concurrent} конкурентных")  # информация

        latencies = []                               # латентности
        errors = 0                                   # ошибки
        status_codes = defaultdict(int)              # коды ответов

        start_time = time.time()                     # время начала

        for i in range(n_requests):                  # отправляем запросы
            try:                                     # обрабатываем ошибки
                features = np.random.randn(20).tolist()  # случайные признаки
                req_start = time.time()               # замеряем время запроса
                resp = requests.post(                 # POST /predict
                    f"{self.base_url}/predict",
                    json={"features": features},
                    timeout=5                        # таймаут 5 секунд
                )
                req_latency = (time.time() - req_start) * 1000  # латентность в мс

                latencies.append(req_latency)         # сохраняем
                status_codes[resp.status_code] += 1   # считаем коды

                if resp.status_code != 200:           # если не 200
                    errors += 1                       # увеличиваем ошибки

            except Exception as e:                    # если ошибка
                errors += 1                           # увеличиваем ошибки
                latencies.append(5000)                # таймаут = 5000ms
                status_codes[500] += 1                # считаем как 500

        total_time = time.time() - start_time        # общее время

        # Вычисляем метрики
        metrics = {                                  # метрики
            'n_requests': n_requests,                # количество запросов
            'total_time_s': round(total_time, 2),    # общее время
            'rps': round(n_requests / total_time, 1),  # запросов в секунду
            'avg_latency_ms': round(np.mean(latencies), 2),  # средняя латентность
            'p50_latency_ms': round(np.percentile(latencies, 50), 2),  # медиана
            'p95_latency_ms': round(np.percentile(latencies, 95), 2),  # p95
            'p99_latency_ms': round(np.percentile(latencies, 99), 2),  # p99
            'max_latency_ms': round(max(latencies), 2),  # максимальная
            'error_rate': round(errors / n_requests * 100, 2),  # процент ошибок
            'status_codes': dict(status_codes),       # коды ответов
        }

        self.results.append(metrics)                 # сохраняем результаты
        return metrics                               # возвращаем метрики

    def print_results(self, metrics):                # выводим результаты
        print(f"\n{'='*40}")                        # разделитель
        print(f"РЕЗУЛЬТАТЫ НАГРУЗОЧНОГО ТЕСТА")      # заголовок
        print(f"{'='*40}")                           # разделитель
        print(f"Запросов: {metrics['n_requests']}")  # количество
        print(f"Общее время: {metrics['total_time_s']}s")  # время
        print(f"RPS: {metrics['rps']}")              # запросов в секунду
        print(f"Средняя латентность: {metrics['avg_latency_ms']}ms")  # средняя
        print(f"p50: {metrics['p50_latency_ms']}ms")  # медиана
        print(f"p95: {metrics['p95_latency_ms']}ms")  # p95
        print(f"p99: {metrics['p99_latency_ms']}ms")  # p99
        print(f"Макс: {metrics['max_latency_ms']}ms")  # макс
        print(f"Error rate: {metrics['error_rate']}%")  # ошибки
        print(f"Коды ответов: {metrics['status_codes']}")  # коды

# Создаём и запускаем нагрузочный тестер
tester = LoadTester(base_url)                        # создаём тестер

# Тест 1: 50 запросов
metrics_50 = tester.run_test(n_requests=50)          # 50 запросов
tester.print_results(metrics_50)                     # выводим результаты

# Тест 2: 200 запросов
metrics_200 = tester.run_test(n_requests=200)        # 200 запросов
tester.print_results(metrics_200)                    # выводим результаты

In [ ]:
# ============================================================
# ШАГ 11 (продолжение): Визуализация результатов нагрузочного тестирования
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))    # 4 подграфика

# Собираем данные из всех тестов
all_latencies = []                                   # все латентности
test_labels = []                                     # метки тестов

for n_req in [50, 100, 200]:                         # разные нагрузки
    try:                                             # обрабатываем ошибки
        latencies = []                               # латентности
        for _ in range(n_req):                       # отправляем запросы
            features = np.random.randn(20).tolist()  # случайные признаки
            start = time.time()                      # замеряем время
            resp = requests.post(                    # POST /predict
                f"{base_url}/predict",
                json={"features": features},
                timeout=5                            # таймаут
            )
            latency = (time.time() - start) * 1000  # латентность
            latencies.append(latency)                # сохраняем
        all_latencies.append(latencies)              # добавляем
        test_labels.append(f'{n_req} запросов')      # метка
    except Exception as e:                           # если ошибка
        print(f"Ошибка при {n_req} запросах: {e}")  # сообщение

# --- 1. Распределение латентности ---
ax = axes[0, 0]                                     # первый подграфик
colors_load = ['#4CAF50', '#2196F3', '#F44336']     # цвета
for i, (lats, label) in enumerate(zip(all_latencies, test_labels)):  # для каждого теста
    ax.hist(lats, bins=30, alpha=0.5, color=colors_load[i], label=label, edgecolor='black')  # гистограмма
ax.set_xlabel('Латентность (мс)', fontsize=11)      # подпись оси
ax.set_ylabel('Количество', fontsize=11)            # подпись оси
ax.set_title('Распределение латентности', fontsize=13, fontweight='bold')  # заголовок
ax.legend(fontsize=10)                              # легенда

# --- 2. RPS vs нагрузка ---
ax = axes[0, 1]                                     # второй подграфик
rps_values = [len(lats) / (sum(lats) / 1000) for lats in all_latencies]  # RPS
ax.bar(test_labels, rps_values, color=colors_load[:len(test_labels)], edgecolor='black')  # столбцы
for i, rps in enumerate(rps_values):                # добавляем значения
    ax.text(i, rps + 0.5, f'{rps:.1f}', ha='center', fontweight='bold')  # текст
ax.set_ylabel('RPS (запросов/сек)', fontsize=11)    # подпись оси
ax.set_title('Пропускная способность', fontsize=13, fontweight='bold')  # заголовок

# --- 3. Перцентили латентности ---
ax = axes[1, 0]                                     # третий подграфик
percentiles = ['p50', 'p90', 'p95', 'p99']          # перцентили
x_pos = np.arange(len(percentiles))                 # позиции
width = 0.25                                        # ширина столбца
for i, (lats, label) in enumerate(zip(all_latencies, test_labels)):  # для каждого теста
    p_values = [np.percentile(lats, int(p[1:])) for p in percentiles]  # значения
    ax.bar(x_pos + i * width, p_values, width, label=label,  # столбцы
           color=colors_load[i], edgecolor='black', alpha=0.8)  # стиль
ax.set_xticks(x_pos + width)                        # позиции меток
ax.set_xticklabels(percentiles, fontsize=11)        # метки
ax.set_ylabel('Латентность (мс)', fontsize=11)      # подпись оси
ax.set_title('Перцентили латентности', fontsize=13, fontweight='bold')  # заголовок
ax.legend(fontsize=10)                              # легенда

# --- 4. Латентность во времени ---
ax = axes[1, 1]                                     # четвёртый подграфик
if len(all_latencies) > 0:                          # если есть данные
    lats = all_latencies[-1]                        # последний тест
    ax.plot(lats, color='#2196F3', alpha=0.5, linewidth=0.8)  # линия
    window = min(10, len(lats))                     # размер окна
    if len(lats) >= window:                         # если достаточно данных
        moving_avg = np.convolve(lats, np.ones(window)/window, mode='valid')  # скользящее среднее
        ax.plot(range(window-1, len(lats)), moving_avg, color='#F44336', linewidth=2, label='Скользящее среднее')  # рисуем
    ax.set_xlabel('Номер запроса', fontsize=11)     # подпись оси
    ax.set_ylabel('Латентность (мс)', fontsize=11)  # подпись оси
    ax.set_title(f'Латентность во времени ({test_labels[-1]})', fontsize=13, fontweight='bold')  # заголовок
    ax.legend(fontsize=10)                          # легенда

plt.suptitle('Результаты нагрузочного тестирования', fontsize=16, fontweight='bold')  # общий заголовок
plt.tight_layout()                                    # подгоняем layout
plt.show()                                            # отображаем

---
## Шаг 12. Ландшафт MLOps-инструментов

Существует множество инструментов для разных этапов MLOps. Разберём основные.

### Категории инструментов

| Категория | Инструменты | Зачем |
|-----------|-------------|-------|
| **Эксперименты** | MLflow, Weights & Biases, Neptune | Отслеживание экспериментов |
| **Версионирование данных** | DVC, LakeFS, Delta Lake | Версии датасетов |
| **Оркестрация** | Airflow, Kubeflow, Prefect | Пайплайны данных |
| **Сервинг** | TF Serving, TorchServe, Triton | Сервинг моделей |
| **Мониторинг** | Evidently, WhyLabs, Arize | Дрифт и метрики |
| **CI/CD** | GitHub Actions, Jenkins, GitLab CI | Автоматизация |

In [ ]:
# ============================================================
# ШАГ 12: Визуализация ландшафта MLOps-инструментов
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 8))    # 2 подграфика

# --- Левый: Radar chart сравнения MLOps платформ ---
ax = axes[0]                                         # первый подграфик

# Платформы и их оценки по категориям
platforms = {                                        # платформы
    'MLflow': {                                      # MLflow
        'Эксперименты': 9, 'Версионирование': 7, 'Деплой': 7,
        'Мониторинг': 4, 'Оркестрация': 5, 'Open Source': 10
    },
    'Kubeflow': {                                    # Kubeflow
        'Эксперименты': 6, 'Версионирование': 5, 'Деплой': 9,
        'Мониторинг': 7, 'Оркестрация': 9, 'Open Source': 8
    },
    'Weights & Biases': {                            # W&B
        'Эксперименты': 10, 'Версионирование': 6, 'Деплой': 4,
        'Мониторинг': 8, 'Оркестрация': 3, 'Open Source': 3
    },
}

categories = list(list(platforms.values())[0].keys())  # категории
n_cats = len(categories)                             # количество категорий
angles = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()  # углы
angles += angles[:1]                                 # замыкаем многоугольник

colors_plat = ['#4CAF50', '#2196F3', '#F44336']     # цвета платформ

for i, (name, scores) in enumerate(platforms.items()):  # для каждой платформы
    values = list(scores.values())                   # значения
    values += values[:1]                             # замыкаем
    ax.plot(angles, values, 'o-', linewidth=2, color=colors_plat[i], label=name, markersize=8)  # линия
    ax.fill(angles, values, alpha=0.1, color=colors_plat[i])  # заливка

ax.set_xticks(angles[:-1])                           # метки
ax.set_xticklabels(categories, fontsize=9)           # названия
ax.set_ylim(0, 11)                                   # пределы
ax.set_title('Сравнение MLOps платформ', fontsize=14, fontweight='bold')  # заголовок
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)  # легенда

# --- Правый: Матрица инструментов по этапам ---
ax = axes[1]                                         # второй подграфик
ax.axis('off')                                       # скрываем оси

# Этапы и инструменты
stages_tools = [                                     # этапы и инструменты
    ('Сбор данных', ['Apache Airflow', 'Prefect', 'Dagster'], '#2196F3'),
    ('Подготовка', ['DVC', 'Great Expectations', 'Pandas'], '#4CAF50'),
    ('Обучение', ['MLflow', 'W&B', 'Neptune', 'Ray'], '#FF9800'),
    ('Оценка', ['Evidently', 'NannyML', 'WhyLabs'], '#9C27B0'),
    ('Деплой', ['TorchServe', 'TF Serving', 'Triton', 'Seldon'], '#F44336'),
    ('Мониторинг', ['Prometheus', 'Grafana', 'Arize'], '#00BCD4'),
]

y_start = 0.95                                       # начальная позиция y
for stage, tools, color in stages_tools:             # для каждого этапа
    # Рисуем название этапа
    rect = plt.Rectangle((0.02, y_start - 0.02), 0.22, 0.06,  # прямоугольник
                         transform=ax.transAxes,     # в координатах осей
                         facecolor=color, alpha=0.7, edgecolor='black')  # стиль
    ax.add_patch(rect)                                # добавляем
    ax.text(0.13, y_start + 0.01, stage,             # текст этапа
            transform=ax.transAxes, ha='center', va='center',  # выравнивание
            fontsize=10, fontweight='bold', color='white')  # стиль

    # Рисуем инструменты
    tools_text = ' | '.join(tools)                   # объединяем инструменты
    ax.text(0.28, y_start + 0.01, tools_text,        # текст инструментов
            transform=ax.transAxes, va='center',     # выравнивание
            fontsize=9, family='monospace')          # стиль

    y_start -= 0.1                                   # сдвигаем вниз

ax.set_title('Инструменты по этапам MLOps', fontsize=14, fontweight='bold')  # заголовок
ax.set_xlim(0, 1)                                    # пределы
ax.set_ylim(0, 1)                                    # пределы

plt.suptitle('Ландшафт MLOps-инструментов', fontsize=16, fontweight='bold')  # общий заголовок
plt.tight_layout()                                    # подгоняем layout
plt.show()                                            # отображаем

In [ ]:
# ============================================================
# ШАГ 12 (продолжение): Сравнительная таблица MLOps инструментов
# ============================================================

# Создаём сравнительную таблицу
tools_comparison = [                                 # сравнительная таблица
    {'Инструмент': 'MLflow', 'Категория': 'Эксперименты + Деплой',  # MLflow
     'Язык': 'Python', 'Сложность': 'Низкая', 'Популярность': 90},
    {'Инструмент': 'Kubeflow', 'Категория': 'Полный MLOps',         # Kubeflow
     'Язык': 'Python + YAML', 'Сложность': 'Высокая', 'Популярность': 75},
    {'Инструмент': 'Weights & Biases', 'Категория': 'Эксперименты',  # W&B
     'Язык': 'Python', 'Сложность': 'Низкая', 'Популярность': 85},
    {'Инструмент': 'DVC', 'Категория': 'Версионирование данных',     # DVC
     'Язык': 'Python + CLI', 'Сложность': 'Средняя', 'Популярность': 70},
    {'Инструмент': 'Airflow', 'Категория': 'Оркестрация',            # Airflow
     'Язык': 'Python', 'Сложность': 'Средняя', 'Популярность': 88},
    {'Инструмент': 'TorchServe', 'Категория': 'Сервинг моделей',     # TorchServe
     'Язык': 'Python + Java', 'Сложность': 'Средняя', 'Популярность': 65},
    {'Инструмент': 'Evidently', 'Категория': 'Мониторинг',           # Evidently
     'Язык': 'Python', 'Сложность': 'Низкая', 'Популярность': 60},
    {'Инструмент': 'Triton', 'Категория': 'Сервинг моделей',         # Triton
     'Язык': 'C++ + Python', 'Сложность': 'Высокая', 'Популярность': 72},
]

# Визуализация популярности
fig, ax = plt.subplots(1, 1, figsize=(14, 6))      # фигура

tools_names = [t['Инструмент'] for t in tools_comparison]  # названия
popularities = [t['Популярность'] for t in tools_comparison]  # популярность
complexities = [t['Сложность'] for t in tools_comparison]  # сложность

# Цвета по сложности
comp_colors = {'Низкая': '#4CAF50', 'Средняя': '#FF9800', 'Высокая': '#F44336'}  # цвета
bar_colors = [comp_colors[c] for c in complexities]  # цвета столбцов

bars = ax.barh(tools_names, popularities, color=bar_colors, edgecolor='black')  # столбцы
for bar, pop, comp in zip(bars, popularities, complexities):  # добавляем метки
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,  # позиция
            f'{pop} ({comp})', va='center', fontsize=11, fontweight='bold')  # текст

ax.set_xlim(0, 105)                                 # пределы x
ax.set_xlabel('Индекс популярности', fontsize=12)   # подпись оси
ax.set_title('MLOps инструменты: популярность и сложность', fontsize=14, fontweight='bold')  # заголовок

# Легенда сложности
from matplotlib.patches import Patch                 # для легенды
legend_elements = [                                  # элементы легенды
    Patch(facecolor='#4CAF50', edgecolor='black', label='Низкая сложность'),  # низкая
    Patch(facecolor='#FF9800', edgecolor='black', label='Средняя сложность'),  # средняя
    Patch(facecolor='#F44336', edgecolor='black', label='Высокая сложность'),  # высокая
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)  # легенда

plt.tight_layout()                                    # подгоняем layout
plt.show()                                            # отображаем

---
## Шаг 13. Практические задания

Выполните эти задания для закрепления навыков MLOps.

### Задание 1: Канареечный деплой
- Реализуйте канареечное развёртывание: 5% трафика на новую модель
- Мониторьте метрики в течение 100 запросов
- Автоматически переключайте 100% трафика, если error rate < 1%

### Задание 2: Система алертов
- Создайте систему алертов с порогами: p99 > 200ms, error rate > 1%, confidence < 0.5
- Отправляйте "алерт" в вывод ноутбука при превышении порога
- Реализуйте эскалацию: warning -> critical -> emergency

### Задание 3: Продвинутый мониторинг
- Добавьте отслеживание feature importance в реальном времени
- Реализуйте детекцию аномалий во входных данных
- Создайте автоматический отчёт о здоровье модели

### Задание 4: Blue-Green деплой
- Реализуйте Blue-Green развёртывание с двумя серверами
- Переключайте трафик между серверами мгновенно
- Добавьте возможность быстрого отката

### Задание 5: ML Pipeline
- Соберите полный пайплайн: данные -> обучение -> оценка -> деплой -> мониторинг
- Реализуйте автоматический триггер переобучения при дрифте
- Добавьте логирование каждого этапа в файл

---

# ИТОГИ КУРСА: Путь от основ до продакшена

Поздравляем! Вы прошли весь путь из **15 ноутбуков** - от основ PyTorch до MLOps в продакшене.

### Ваше путешествие

| # | Ноутбук | Ключевая тема | Главный навык |
|---|---------|---------------|---------------|
| 01 | **PyTorch Intro** | Тензоры, autograd, градиентный спуск | Основы фреймворка |
| 02 | **What is ML** | Типы обучения, функции потерь, переобучение | Интуиция ML |
| 03 | **Notebook to Service** | От кода к API, FastAPI, Docker | Сервинг моделей |
| 04 | **CNN** | Свёртки, пулинг, архитектуры | Компьютерное зрение |
| 05 | **Text Embeddings** | Word2Vec, GloVe, эмбеддинги | Представление текста |
| 06 | **Classic ML / RNN-LSTM** | Рекуррентные сети, последовательности | Работа с рядами |
| 07 | **Attention & Transformer** | Механизм внимания, self-attention | Архитектура Transformer |
| 08 | **BERT** | Предобучение, файн-тюнинг, MLM | Языковые модели |
| 09 | **GPT** | Авторегрессия, генерация текста | Генеративные модели |
| 10 | **Finetune LLM** | LoRA, QLoRA, PEFT, датасеты | Файн-тюнинг LLM |
| 11 | **RLHF** | Reinforcement Learning, награды, PPO | Выравнивание моделей |
| 12 | **RAG** | Поиск, индексация, генерация с контекстом | RAG-системы |
| 13 | **Multimodal** | CLIP, VLM, изображение + текст | Мультимодальность |
| 14 | **GAN** | Генеративные сети, DCGAN, cGAN | Генерация изображений |
| 15 | **MLOps** | Сериализация, деплой, мониторинг, дрифт | Продакшен ML |

### Что вы умеете после курса

**Фундамент:**
- Понимаете, как работают нейросети "под капотом"
- Умеете реализовывать модели с нуля в PyTorch
- Знаете математику: градиенты, функции потерь, оптимизация

**Архитектуры:**
- CNN для компьютерного зрения
- RNN/LSTM для последовательностей
- Transformer - корень всех современных LLM
- BERT и GPT - два направления языковых моделей
- GAN для генерации изображений

**Продвинутые техники:**
- Файн-тюнинг LLM (LoRA, QLoRA)
- RLHF для выравнивания моделей
- RAG для работы с внешними знаниями
- Мультимодальные модели (текст + изображения)

**Продакшен:**
- Сериализация и версионирование моделей
- FastAPI серверы для ML
- A/B тестирование моделей
- Мониторинг и детекция дрифта
- Автоматическое переобучение
- Нагрузочное тестирование

### Следующие шаги

1. **Углубитесь в специализацию** - выберите CV, NLP или RecSys
2. **Изучите MLOps платформы** - MLflow, Kubeflow, W&B
3. **Практикуйтесь на Kaggle** - реальные задачи и соревнования
4. **Читайте papers** - ArXiv, Papers With Code, Hugging Face Blog
5. **Создайте портфолио** - опубликуйте проекты на GitHub
6. **Вступите в сообщество** - AI-чатбы, конференции, митапы

### Ресурсы для дальнейшего изучения

| Ресурс | Ссылка | Что даёт |
|--------|--------|----------|
| **Hugging Face** | huggingface.co | Модели, датасеты, Spaces |
| **Papers With Code** | paperswithcode.com | Статьи + код + бенчмарки |
| **MLflow** | mlflow.org | Эксперименты и деплой |
| **FastAPI** | fastapi.tiangolo.com | Документация API |
| **PyTorch** | pytorch.org | Документация и туториалы |
| **ArXiv** | arxiv.org | Свежие исследования |

> **Помните:** вы - автор, а не потребитель. Ломайте, чините, понимайте. Никаких чёрных ящиков. Этот принцип - ваш главный инструмент на пути в ML.

### Спасибо за путь!

Вы прошли путь от `torch.tensor([1.0])` до продакшен-системы с мониторингом, A/B тестированием и автопереобучением. Это серьёзное достижение. Продолжайте строить, экспериментировать и учиться!